In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
syedyasir_structured_abstracts_path = kagglehub.dataset_download('syedyasir/structured-abstracts')

print('Data source import complete.')


Using Colab cache for faster access to the 'structured-abstracts' dataset.
Data source import complete.


In [2]:
 # This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/structured-abstracts/arxiv_structured.parquet


In [3]:
#All the Imports are done here

import wandb
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import sys
import os
import nltk
from transformers import AutoTokenizer
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import torch.nn.utils as utils
from tqdm import tqdm
from transformers import GPT2Model, GPT2Config
from sklearn.model_selection import train_test_split
from bs4 import BeautifulSoup
from transformers import DistilBertModel, DistilBertConfig
from transformers import DistilBertTokenizer
import matplotlib.pyplot as plt
import seaborn as sns
import re
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, DistilBertTokenizer
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from sklearn.neighbors import NearestNeighbors
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from nltk.tokenize import sent_tokenize
from pathlib import Path
from transformers import DynamicCache
import torch
from contextlib import contextmanager

In [4]:
wandb.login(key="0ce56922c7ea30310a87d49246b15bc7d7ca9c89")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yasir-alam14 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
df = pd.read_parquet(syedyasir_structured_abstracts_path)
df.head()

,id,abstract,num_sentences,structured_abstract
0,704.0009,We discuss the results from the combined IRAC ...,10,<BOS> We discuss the results from the combined...
1,704.0017,Results from spectroscopic observations of the...,7,<BOS> Results from spectroscopic observations ...
2,704.0022,We present Lie group integrators for nonlinear...,8,<BOS> We present Lie group integrators for non...
3,704.0059,We derive masses and radii for both components...,9,<BOS> We derive masses and radii for both comp...
4,704.0066,Possible (algebraic) commutation relations in ...,8,<BOS> Possible (algebraic) commutation relatio...


In [6]:
df['num_sentences'].describe()

,num_sentences
count,401264.000000
mean,8.369181
std,1.052987
min,7.000000
25%,7.000000
50%,8.000000
75%,9.000000
max,10.000000


In [7]:
df["structured_abstract"] = df["structured_abstract"].str.replace(
    "<endoftext>",
    "<|endoftext|>",
    regex=False
)

In [8]:
df['structured_abstract'].iloc[0]

"<BOS> We discuss the results from the combined IRAC and MIPS c2d Spitzer Legacy observations of the Serpens star-forming region. <EOS> <BOS> In particular we present a set of criteria for isolating bona fide young stellar objects, YSO's, from the extensive background contamination by extra-galactic objects. <EOS> <BOS> We then discuss the properties of the resulting high confidence set of YSO's. <EOS> <BOS> We find 235 such objects in the 0.85 deg^2 field that was covered with both IRAC and MIPS. <EOS> <BOS> An additional set of 51 lower confidence YSO's outside this area is identified from the MIPS data combined with 2MASS photometry. <EOS> <BOS> We describe two sets of results, color-color diagrams to compare our observed source properties with those of theoretical models for star/disk/envelope systems and our own modeling of the subset of our objects that appear to be star+disks. <EOS> <BOS> These objects exhibit a very wide range of disk properties, from many that can be fit with 

In [9]:
train_df, valid_df = train_test_split(df, test_size=0.01, random_state=42)

print(f"Train shape: {train_df.shape}")
print(f"Valid shape: {valid_df.shape}")

Train shape: (397251, 4)
Valid shape: (4013, 4)


In [10]:
# the encoder network for the HVAE
class MLPNetwork(nn.Module):
    """
    Optimized MLP for HVAE Encoder q(z|x).
    1. Removes bottlenecks (Maintains width).
    2. Uses GELU (Matches BERT/Transformer activations).
    3. Implements Near-Zero Initialization to prevent early KL shock.
    """
    def __init__(self, input_dim, latent_dim):
        super().__init__()

        # 1. Maintain Width: Do not compress to //2 or //4 immediately.
        # We want deep non-linearities, not compression.
        hidden_dim = input_dim

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.ln1 = nn.LayerNorm(hidden_dim)

        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)

        # Output layer
        self.fc_out = nn.Linear(hidden_dim, 2 * latent_dim, bias=False)

        # Match activation to DistilBERT/GPT-2 (GELU is smoother than ReLU)
        self.activation = nn.GELU()

        # --- CRITICAL: Near-Zero Initialization ---
        # This ensures that at Step 0, the posterior q(z|x) is very close to N(0,1).
        # This prevents the initial KL loss from being huge, which scares the
        # optimizer into "killing" the latent variable immediately (Collapse).
        torch.nn.init.normal_(self.fc_out.weight, mean=0.0, std=0.001)


    def forward(self, h):
        # Post-Norm architecture (standard for Transformers)
        x = self.fc1(h)
        x = self.ln1(x)
        x = self.activation(x)

        x = self.fc2(x)
        x = self.ln2(x)
        x = self.activation(x)

        output = self.fc_out(x)

        mu, raw_var_score = output.chunk(2, dim=-1)

        # Robust Softplus
        sigma2 = F.softplus(raw_var_score) + 1e-6

        return mu, sigma2

class HT_HVAE_InferenceNetwork(nn.Module):
    """
    The Hierarchical Transformer Encoder (Inference Network) q(z|x).
    Now uses a randomly initialized Transformer for word-level encoding.
    """
    def __init__(self, hyperparams):
        super().__init__()

        self.latent_dim = hyperparams['latent_dim']
        self.d_model = hyperparams['d_model']
        self.vocab_size = hyperparams['vocab_size'] # Now the GPT-2 vocab size
        self.max_sentences = hyperparams['max_sentences']
        self.max_words = hyperparams['max_words']
        self.n_heads = hyperparams['encoder_heads']
        self.dropout = hyperparams['encoder_dropout']
        self.n_layers = hyperparams['encoder_layers']
        self.pad_idx = hyperparams['pad_index'] # Ensure this is passed in hyperparams

        # --- 2.1. Word-Level Transformer Encoder (Random Init) ---

        # 1. Token Embeddings (Replaces DistilBERT embeddings)
        self.word_embedding = nn.Embedding(self.vocab_size, self.d_model, padding_idx=self.pad_idx)

        # 2. Positional Embeddings (Required since we aren't using pre-trained BERT)
        self.word_position_embedding = nn.Embedding(self.max_words, self.d_model)

        # 3. Transformer Encoder Stack
        word_encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.d_model,
            nhead=self.n_heads,
            dim_feedforward=4 * self.d_model,
            dropout=self.dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True
        )
        self.word_encoder = nn.TransformerEncoder(word_encoder_layer, num_layers=self.n_layers)

        # --- 2.2. Sentence-Level Transformer Encoder ---

        self.sentence_position_embedding = nn.Embedding(self.max_sentences + 1, self.d_model)

        sentence_encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.d_model,
            nhead=self.n_heads,
            dim_feedforward=4 * self.d_model,
            dropout=self.dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True
        )
        self.transformer_sentence = nn.TransformerEncoder(sentence_encoder_layer, num_layers=self.n_layers)

        # --- 2.3. Latent Variable Networks (MLP) ---

        # MLP for Global Posterior q(zt | x)
        self.mlp_global = MLPNetwork(self.d_model, self.latent_dim)
        self.local_latent = self.latent_dim

        # MLP for Local Posterior q(zi | xi)
        self.mlp_local = MLPNetwork(self.d_model, self.local_latent)

        # Learnable Document Token (similar to CLS for the whole document)
        self.doc_token = nn.Parameter(torch.randn(1, 1, self.d_model))

    def forward(self, input_ids, word_mask):
        """
        Args:
            input_ids: (batch_size, MAX_SENTENCES, MAX_WORDS) -> Pass dec_input_ids here
            word_mask: (batch_size, MAX_SENTENCES, MAX_WORDS) -> Pass dec_word_mask here
        """
        batch_size, max_sentences, max_words = input_ids.shape

        # --- 2.1. Word-Level Encoding (Shared Random Init Transformer) ---

        # Flatten: (B * N, W)
        flat_input_ids = input_ids.view(-1, max_words)

        # Create Boolean Mask for Transformer (True where padding exists)
        # word_mask is 1 for real, 0 for pad. Transformer needs True for Pad.
        flat_key_padding_mask = (word_mask.view(-1, max_words) == 0)

        # 1. Get Embeddings
        token_embeds = self.word_embedding(flat_input_ids) # (B*N, W, D)

        # 2. Add Positional Embeddings
        positions = torch.arange(0, max_words, device=input_ids.device).unsqueeze(0)
        pos_embeds = self.word_position_embedding(positions) # (1, W, D)

        hidden_states = token_embeds + pos_embeds

        # 3. Pass through Transformer Encoder
        # src_key_padding_mask shape: (B*N, W)
        word_encoder_output = self.word_encoder(
            src=hidden_states,
            src_key_padding_mask=flat_key_padding_mask
        )

        # 4. Extract Sentence Representation (Hw_0)
        # Since input is GPT-2 style with <BOS> at index 0, we take the 0th vector
        # This vector now contains aggregated info from the whole sentence via self-attention
        H_w_0_flat = word_encoder_output[:, 0, :] # (B*N, D)

        # Handle purely padding sentences (firewall)
        is_padding_sentence = flat_key_padding_mask[:, 0]
        H_w_0_flat = H_w_0_flat.masked_fill(is_padding_sentence.unsqueeze(1), 0.0)

        # Reshape back: (batch_size, MAX_SENTENCES, D)
        H_w_0 = H_w_0_flat.view(batch_size, max_sentences, self.d_model)

        # --- 2.3. Local Posterior (q(zi | xi)) ---
        mu_i_flat, sigma2_i_flat = self.mlp_local(H_w_0_flat)
        mu_i = mu_i_flat.view(batch_size, max_sentences, self.local_latent)
        sigma2_i = sigma2_i_flat.view(batch_size, max_sentences, self.local_latent)

        # --- 2.2. Sentence-Level Encoding ---

        # Prepare inputs: Add Doc Token + Sentence Position Embeddings
        batch_doc_token = self.doc_token.expand(batch_size, -1, -1)
        H_sen_input_ = torch.cat([batch_doc_token, H_w_0], dim=1) # (B, N+1, D)

        position_ids = torch.arange(0, max_sentences + 1, device=input_ids.device)
        position_embeddings = self.sentence_position_embedding(position_ids)
        position_embeddings = position_embeddings.unsqueeze(0).expand(batch_size, -1, -1)

        H_sen_input = H_sen_input_ + position_embeddings

        # Create Sentence Mask (True where sentence is padding)
        # sentence padding is where the first word is padded
        sentence_pad_mask = (word_mask[:, :, 0] == 0) # (B, N)
        doc_mask = torch.zeros((batch_size, 1), dtype=torch.bool, device=input_ids.device)
        full_sentence_mask = torch.cat([doc_mask, sentence_pad_mask], dim=1)

        # Sentence Transformer
        H_s = self.transformer_sentence(
            src=H_sen_input,
            src_key_padding_mask=full_sentence_mask
        )

        H_s_0 = H_s[:, 0, :] # (B, D)

        # --- 2.3. Global Posterior (q(zt | x)) ---
        mu_t, sigma2_t = self.mlp_global(H_s_0)

        return mu_t, sigma2_t, mu_i, sigma2_i

In [11]:
# The main decoder model for the HVAE
class MLPNetworkForPrior(nn.Module):
    """
    Optimized Prior Network p(z_i | z_t, z_i-1).
    1. Widened layers (No bottleneck).
    2. Context Dropout (Forces reliance on z_t).
    3. Near-Zero Init (Prevents initial KL explosion).
    """
    def __init__(self, latent_dim, context_dropout_rate=0):
        super().__init__()
        self.context_dropout_rate = context_dropout_rate

        # Input: Global (32) + Local (32) = 96
        input_dim = latent_dim + latent_dim

        # 1. Maintain Width:
        # Instead of compressing, we project up or keep equal.
        # 128 gives enough capacity to mix Global and Previous-Local info.
        hidden_dim = 2*input_dim

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.ln1 = nn.LayerNorm(hidden_dim)

        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)

        self.fc_out = nn.Linear(hidden_dim, 2 * latent_dim, bias=False) # Outputs (mu, var_param)

        self.activation = nn.GELU()

        # --- CRITICAL: Near-Zero Initialization ---
        # Initialize output to be very close to N(0, 1) parameters.
        # mu -> 0, sigma_param -> 0 (which becomes softplus(0) ~ 0.69)
        # This matches the initialization of your Encoder.
        torch.nn.init.normal_(self.fc_out.weight, mean=0.0, std=0.001)
        # torch.nn.init.constant_(self.fc_out.bias, 0)

    def forward(self, z_t, z_i_minus_1):
        """
        Args:
            z_t (Tensor): Global latent (B, D_global)
            z_i_minus_1 (Tensor): Previous local latent (B, D_local)
        """

        # --- 2. Context Dropout (The "Firewall") ---
        # Randomly zero out the previous sentence information.
        # This forces the network to look at z_t to guess the current sentence state.
        if self.training and self.context_dropout_rate > 0:
            mask_prob = torch.rand(z_t.shape[0], 1, device=z_t.device)
            # If random > rate, keep signal (1.0). Else drop (0.0).
            keep_mask = (mask_prob > self.context_dropout_rate).float()
            z_i_minus_1 = z_i_minus_1 * keep_mask

        # Concatenate inputs
        h = torch.cat([z_t, z_i_minus_1], dim=-1)

        # Block 1
        x = self.fc1(h)
        x = self.ln1(x)
        x = self.activation(x)

        # Block 2
        x = self.fc2(x)
        x = self.ln2(x)
        x = self.activation(x)

        # Output Head
        output = self.fc_out(x)

        mu, raw_var_score = output.chunk(2, dim=-1)

        # Robust Softplus
        sigma2 = F.softplus(raw_var_score) + 1e-6

        return mu, sigma2

class HT_HVAE_GenerativeNetwork(nn.Module):
    """
    The Hierarchical VAE Generative Network (Decoder) p(x|z).
    Uses GRU for sentence planning and modified GPT-2 for word generation.
    """
    def __init__(self, hyperparams):
        super().__init__()

        self.latent_dim = hyperparams['latent_dim']
        self.local_latent = self.latent_dim
        self.d_model = hyperparams['d_model']
        self.gpt2_model_name = hyperparams['gpt2_model_name']
        self.vocab_size = hyperparams['vocab_size']
        self.gru_layers = hyperparams['gru_layers']

        # 3.2. Local Prior Network
        self.prior_mlp = MLPNetworkForPrior(self.latent_dim)
        self.gru_initial_projection = nn.Linear(self.latent_dim, self.d_model)

        # 3.3. Sentence-Level Decoder (GRU)
        self.gru = nn.GRU(
          input_size=self.local_latent + self.latent_dim,
          hidden_size=self.d_model,
          num_layers=self.gru_layers,
          batch_first=True
        )

        # 3.4. Load GPT-2
        config = GPT2Config.from_pretrained(self.gpt2_model_name)
        self.gpt2_model = GPT2Model.from_pretrained(self.gpt2_model_name, config=config)
        self.gpt2_model.resize_token_embeddings(self.vocab_size)
        gpt2_hidden_size = self.gpt2_model.config.n_embd

        # Resize embeddings to your vocab
        self.gpt2_model.resize_token_embeddings(self.vocab_size)
        gpt2_hidden_size = self.gpt2_model.config.n_embd

        cfg = self.gpt2_model.config
        self.gpt2_n_layer = cfg.n_layer
        self.gpt2_n_head  = cfg.n_head
        self.gpt2_n_embd  = cfg.n_embd
        assert self.gpt2_n_embd % self.gpt2_n_head == 0
        self.gpt2_head_dim = self.gpt2_n_embd // self.gpt2_n_head

        # choose prefix length for KV memory
        self.prefix_len = getattr(self, "prefix_len", 5)  # or pass as arg to __init__
        # (Optional but often helps) normalize plan before projections
        self.plan_ln = nn.LayerNorm(self.d_model)
        self.plan_to_emb = nn.Linear(self.d_model, self.gpt2_n_embd, bias=False)


        self.plan_gate = nn.Sequential(
                nn.Linear(self.d_model, 1),
                nn.Sigmoid()
        )

        hidden = 4 * self.gpt2_n_embd  # you can tune (2x, 4x, 8x)
        self.plan_to_kv = nn.Sequential(
            nn.Linear(self.d_model, hidden),
            nn.Tanh(),
            nn.Linear(hidden, self.gpt2_n_layer * 2 * self.gpt2_n_head * self.prefix_len * self.gpt2_head_dim),
        )

        # No need to concat plan here anymore; the prefix handles the "Mood"
        self.final_linear = nn.Linear(gpt2_hidden_size, self.vocab_size, bias=False)

        # Hyperparameters
        self.word_dropout_rate = hyperparams['word_dropout_rate']
        self.mask_token_id = hyperparams.get('mask_token_id', self.gpt2_model.config.eos_token_id)




    def forward(self, input_ids, word_mask, z_t, z_i_samples, mu_i_prior=None, log_sigma2_i_prior=None):
        batch_size, max_sentences, max_words = input_ids.shape

        # --- 1. GRU Plan Generation ---
        # Initialize GRU with Global Latent z_t
        max_sentences = z_i_samples.size(1)
        z_t_expanded = z_t.unsqueeze(1).repeat(1, max_sentences, 1)

        # 2. Concatenate: GRU Input = [Local Latent; Global Latent]
        # Result shape: (Batch, Num_Sentences, Local_Dim + Global_Dim)
        gru_input = torch.cat([z_i_samples, z_t_expanded], dim=-1)

        # 3. Initialize Hidden State (Optional, but good to keep)
        h_0_projected = self.gru_initial_projection(z_t).unsqueeze(0)
        h_0 = h_0_projected.repeat(self.gru.num_layers, 1, 1)

        # 4. Pass Concatenated Input to GRU
        plan_vectors, _ = self.gru(gru_input, h_0)

        # --- 2. Prior Network (KL Logic unchanged) ---
        z_i_minus_1 = torch.cat([torch.zeros_like(z_i_samples[:, :1, :]), z_i_samples[:, :-1, :]], dim=1).view(-1, self.local_latent)
        z_t_flat = z_t.unsqueeze(1).repeat(1, max_sentences, 1).view(-1, self.latent_dim)
        mu_i_prior, sigma2_i_prior = self.prior_mlp(z_t_flat, z_i_minus_1)
        mu_i_prior = mu_i_prior.view(batch_size, max_sentences, self.local_latent)
        sigma2_i_prior = sigma2_i_prior.view(batch_size, max_sentences, self.local_latent)

        # --- 3. GENERATE PREFIXES (The New "Deep Injection") ---
        # Flatten plans: (B*N, D_MODEL)
        flat_plan_vectors = plan_vectors.reshape(-1, self.d_model)

        # Generate Virtual Tokens: (B*N, Prefix_Len, GPT_Hidden)
        # These are now normalized and scaled correctly by your class

        # --- 4. PREPARE TEXT INPUT ---
        flat_input_ids = input_ids.view(-1, max_words)
        bsz = flat_input_ids.size(0)


        # Word Dropout Logic
        if self.training and self.word_dropout_rate > 0:
            rand_mask = torch.rand(flat_input_ids.shape, device=input_ids.device)
            flat_word_mask_bool = word_mask.view(-1, max_words).bool()
            drop_mask = (rand_mask < self.word_dropout_rate) & flat_word_mask_bool
            input_ids_for_decoder = flat_input_ids.clone()
            input_ids_for_decoder[drop_mask] = self.mask_token_id
        else:
            input_ids_for_decoder = flat_input_ids



        tok_emb = self.gpt2_model.wte(input_ids_for_decoder)
        flat_plan_vectors = self.plan_ln(flat_plan_vectors)
        plan_emb = self.plan_to_emb(flat_plan_vectors)
        gate = self.plan_gate(flat_plan_vectors)
        tok_emb = tok_emb + (gate * plan_emb).unsqueeze(1)

        kv = self.plan_to_kv(flat_plan_vectors)
        kv = kv.view(
        bsz,
        self.gpt2_n_layer,
        2,
        self.gpt2_n_head,
        self.prefix_len,
        self.gpt2_head_dim
    )

        kv = kv.permute(1, 2, 0, 3, 4, 5).contiguous()  # (L, 2, B, H, P, Hd)

        past = DynamicCache()
        for l in range(self.gpt2_n_layer):
            past.update(kv[l, 0], kv[l, 1], l)   # each: (B, H, P, Hd)

        flat_word_mask = word_mask.view(-1, max_words) # (B*N, Max_Words)
        prefix_mask = torch.ones(
            bsz, self.prefix_len,
            device=flat_word_mask.device,
            dtype=flat_word_mask.dtype
        )


        attention_mask = torch.cat([prefix_mask, flat_word_mask], dim=1)



        # --- 7. FORWARD PASS ---
        out = self.gpt2_model(
            inputs_embeds=tok_emb,
            attention_mask=attention_mask,
            past_key_values=past,
            use_cache=False,
            return_dict=True,
    )

        # Get hidden states
        # Shape: (B*N, Prefix_Len + Max_Words, Hidden)
        text_hidden_states = out.last_hidden_state

        # --- 9. LOGITS ---
        # Compute logits only for text
        reconstruction_logits = self.final_linear(text_hidden_states)

        # Reshape back to batch structure
        reconstruction_logits = reconstruction_logits.view(
            batch_size, max_sentences, max_words, self.vocab_size
        )

        return reconstruction_logits, mu_i_prior, sigma2_i_prior

    def generate_autoregressive(
        self,
        z_t,
        z_i_samples,
        target_ids,
        eos_sentence_id,
        eos_doc_id,
        bos_token_id,
        pad_token_id,
        max_words=50,
        prime_len=0
    ):
        """
        Autoregressive generation with latent injection via 'past'.
        Assumes batch_size = 1.
        """
        self.eval()
        batch_size = 1
        batch, max_sentences, target_max_words = target_ids.shape

        # --- 1. REPLICATE PLAN GENERATION (Same as Forward) ---
        # Initialize GRU with Global Latent z_t
        z_t_expanded = z_t.unsqueeze(1).repeat(1, max_sentences, 1)

        # Concatenate: [Local; Global]
        gru_input = torch.cat([z_i_samples, z_t_expanded], dim=-1)

        # Initialize Hidden State
        h_0_projected = self.gru_initial_projection(z_t).unsqueeze(0)
        h_0 = h_0_projected.repeat(self.gru.num_layers, 1, 1)

        # Pass to GRU -> Get Plan Vectors
        # shape: (1, max_sentences, d_model)
        plan_vectors, _ = self.gru(gru_input, h_0)

        # --- 2. GENERATION LOOP ---
        all_generated_ids = [] # Will be list of lists (sentences -> words)
        all_logits = []
        all_predicted_ids = []
        stop_generation_flag = False

        with torch.no_grad():
            # Loop over sentences
            for sent_idx in range(max_sentences):
                if stop_generation_flag:
                    break

                # Get the specific latent plan for this sentence
                # Shape: (1, d_model) -> Needs to be compatible with GPT2 'past'
                # Based on your forward, we pass it directly.
                current_plan = plan_vectors[:, sent_idx, :]

                tok_emb = self.gpt2_model.wte(input_ids_for_decoder)
                current_plan = self.plan_ln(current_plan)
                plan_emb = self.plan_to_emb(current_plan)
                gate = self.plan_gate(current_plan)
                embedding_bias = (gate * plan_emb).unsqueeze(1)

                kv = self.plan_to_kv(current_plan)
                kv = kv.view(
                    1,
                    self.gpt2_n_layer,
                    2,
                    self.gpt2_n_head,
                    self.prefix_len,
                    self.gpt2_head_dim
                )

                kv = kv.permute(1, 2, 0, 3, 4, 5).contiguous()  # (L, 2, B, H, P, Hd)

                past = DynamicCache()
                for l in range(self.gpt2_n_layer):
                    past.update(kv[l, 0], kv[l, 1], l)   # each: (B, H, P, Hd)


                # Start the sentence with BOS
                next_input_id = torch.tensor([[bos_token_id]], device=z_t.device)

                sent_generated_ids = [bos_token_id] # Track BOS in output
                sent_logits = []
                predicted_ids = []

                # Loop over words
                is_sent_finished = False
                for word_idx in range(max_words):

                    tok_emb = self.gpt2_model.wte(next_input_id)
                    tok_emb = tok_emb + embedding_bias
                    gpt2_output = self.gpt2_model(
                            inputs_embeds=tok_emb,
                            past_key_values=past,
                            use_cache=True,
                            return_dict=True
                        )

                    past = gpt2_output.past_key_values

                    # Get last hidden state and project to vocab
                    last_hidden_state = gpt2_output.last_hidden_state[:, -1, :]
                    next_token_logits = self.final_linear(last_hidden_state) # (1, 1, Vocab)
                    sent_logits.append(next_token_logits.squeeze(0).squeeze(0)) # Store logits

                    # --- B. Decode & Priming Logic ---
                    predicted_id = torch.argmax(next_token_logits, dim=-1)
                    predicted_ids.append(predicted_id.item())
                    # Determine what the NEXT input will be
                    if word_idx < prime_len:
                        # K-Priming: Force the next input to be from targets
                        # Note: target_ids usually starts with BOS, so target_ids[..., 0] is BOS.
                        # If word_idx=0 (we just processed BOS), we want target_ids[..., 1].
                        # Ensure we don't go out of bounds of target_ids
                        if (word_idx + 1) < target_ids.size(2):
                            next_input_id = target_ids[:, sent_idx, word_idx + 1].view(1, 1)
                            actual_id_to_store = next_input_id.item()
                        else:
                            # Fallback if target is shorter than prime_len
                            next_input_id = predicted_id
                            actual_id_to_store = predicted_id.item()
                    else:
                        # Normal Greedy Decoding
                        next_input_id = predicted_id
                        actual_id_to_store = predicted_id.item()

                    sent_generated_ids.append(actual_id_to_store)
                    next_input_id = torch.tensor(
                            [[actual_id_to_store]],
                            device=z_t.device
                        )

                    # --- C. Stopping Conditions ---

                    if actual_id_to_store == eos_doc_id:
                        stop_generation_flag = True
                        break # Stop words

                    if is_sent_finished:
                        break

                    if actual_id_to_store == eos_sentence_id:
                        is_sent_finished = True # Stop words (go to next sentence)

                # Append this sentence's results
                all_generated_ids.append(sent_generated_ids)
                all_logits.append(sent_logits)
                all_predicted_ids.append(predicted_ids)

        # --- 3. FORMAT OUTPUT ---
        # Because sentences might be different lengths, we usually pad them
        # to return a clean tensor, or return a list.
        # Here I will pad with a padding ID (assuming 0 or eos_doc) to match dimensions.

        # Find max length generated
        max_gen_len = max([len(s) for s in all_generated_ids])
        final_ids_tensor = torch.full((1, len(all_generated_ids), max_gen_len), pad_token_id, dtype=torch.long, device=z_t.device)

        max_pred_len = max([len(s) for s in all_predicted_ids])
        predicted_ids_tensor = torch.full((1, len(all_predicted_ids), max_pred_len), pad_token_id, dtype=torch.long, device=z_t.device)

        for i, s_ids in enumerate(all_predicted_ids):
            length = len(s_ids)
            predicted_ids_tensor[0, i, :length] = torch.tensor(s_ids, device=z_t.device)

        for i, s_ids in enumerate(all_generated_ids):
            length = len(s_ids)
            final_ids_tensor[0, i, :length] = torch.tensor(s_ids, device=z_t.device)

        vocab_size = self.vocab_size # Ensure this attribute exists
        final_logits_tensor = torch.zeros(
            (1, max_sentences, target_max_words, vocab_size),
            dtype=torch.float,
            device=z_t.device
        )

        for i, s_logits in enumerate(all_logits):
            if len(s_logits) > 0:
                # stack the list of tensors into (Seq_Len, Vocab)
                stacked_logits = torch.stack(s_logits)
                length = stacked_logits.size(0)
                # Fill the tensor, truncate if somehow longer than target (handled by loop limit, but safe to slice)
                limit = min(length, target_max_words)
                final_logits_tensor[0, i, :limit, :] = stacked_logits[:limit, :]

        return final_ids_tensor, all_generated_ids, final_logits_tensor, predicted_ids_tensor

In [12]:
#HVAE LOSS
class HT_HVAE_Loss(nn.Module):
    def __init__(self, hyperparameters):
        super().__init__()
        self.vocab_size = hyperparameters["vocab_size"]
        self.pad_idx = hyperparameters["pad_index"]
        self.latent_dim = hyperparameters["latent_dim"]
        self.local_latent = self.latent_dim

        self.cross_entropy_loss = nn.CrossEntropyLoss(
            ignore_index=self.pad_idx, reduction="none"
        )

        # free-bits thresholds (nats per dim)
        self.free_bits_global = 0.5
        self.free_bits_local = 0.2

    def forward(
        self,
        mu_t_q, sigma2_t_q,
        mu_i_q, sigma2_i_q,
        reconstruction_logits,
        mu_i_p, sigma2_i_p,
        target_ids, word_mask,
        local_kl_beta=0.5, global_kl_beta=0.1,
    ):
        eps = 1e-9
        B, S, W = target_ids.shape

        # -------------------------
        # A) Reconstruction: mean over batch (sum tokens per example, then avg over batch)
        # -------------------------
        shift_logits = reconstruction_logits[..., :-1, :].contiguous()  # (B,S,W-1,V)
        shift_labels = target_ids[..., 1:].contiguous()                 # (B,S,W-1)

        nll_flat = self.cross_entropy_loss(
            shift_logits.view(-1, self.vocab_size),
            shift_labels.view(-1)
        )  # (B*S*(W-1),) with PAD ignored as 0

        nll = nll_flat.view(B, S, W - 1)
        token_mask = (shift_labels != self.pad_idx).float()             # (B,S,W-1)

        recon_per_ex = (nll * token_mask).sum(dim=(1, 2))               # (B,)
        reconstruction_loss = recon_per_ex.mean()                       # scalar (avg over batch)

        num_active_tokens = token_mask.sum().float()
        mean_token_loss = recon_per_ex.sum() / (num_active_tokens + eps)  # scalar (avg per token)

        # Sentence mask: 1 for real sentences, 0 for padded sentences
        sentence_mask = (word_mask.sum(dim=-1) > 0).float()             # (B,S)

        # -------------------------
        # B) Global KL: sum dims per example, mean over batch
        # -------------------------
        global_kl_raw = 0.5 * (
            sigma2_t_q + mu_t_q.pow(2) - 1.0 - torch.log(sigma2_t_q + eps)
        )  # (B,D)

        global_frac_dims_clamped = (global_kl_raw < self.free_bits_global).float().mean()

        global_kl_charged = torch.clamp(global_kl_raw, min=self.free_bits_global)  # (B,D)

        global_kl_raw_per_ex = global_kl_raw.sum(dim=-1)              # (B,)
        global_kl_charged_per_ex = global_kl_charged.sum(dim=-1)      # (B,)

        global_kl_raw_mean = global_kl_raw_per_ex.mean()              # scalar
        global_kl_loss = global_kl_charged_per_ex.mean()              # scalar

        # -------------------------
        # C) Local KL: sum dims, sum active sentences per example, mean over batch
        # -------------------------
        mu_p_flat = mu_i_p.reshape(-1, self.local_latent)
        sigma2_p_flat = sigma2_i_p.reshape(-1, self.local_latent)
        mu_q_flat = mu_i_q.reshape(-1, self.local_latent)
        sigma2_q_flat = sigma2_i_q.reshape(-1, self.local_latent)

        term1 = torch.log(sigma2_p_flat + eps) - torch.log(sigma2_q_flat + eps)
        term2 = (sigma2_q_flat + (mu_q_flat - mu_p_flat).pow(2)) / (sigma2_p_flat + eps)
        local_kl_raw_flat = 0.5 * (term1 + term2 - 1.0)               # (B*S,D)

        local_kl_raw = local_kl_raw_flat.view(B, S, self.local_latent)  # (B,S,D)

        active_mask_bsd = sentence_mask.unsqueeze(-1)                   # (B,S,1)
        local_under = (local_kl_raw < self.free_bits_local).float() * active_mask_bsd
        local_frac_dims_clamped = local_under.sum() / (active_mask_bsd.sum() * self.local_latent + eps)

        local_kl_charged = torch.clamp(local_kl_raw, min=self.free_bits_local)  # (B,S,D)

        local_kl_raw_per_sent = local_kl_raw.sum(dim=-1)               # (B,S)
        local_kl_charged_per_sent = local_kl_charged.sum(dim=-1)       # (B,S)

        local_kl_raw_per_ex = (local_kl_raw_per_sent * sentence_mask).sum(dim=-1)        # (B,)
        local_kl_charged_per_ex = (local_kl_charged_per_sent * sentence_mask).sum(dim=-1) # (B,)

        local_kl_raw_mean = local_kl_raw_per_ex.mean()                 # scalar
        local_kl_loss = local_kl_charged_per_ex.mean()                 # scalar

        # -------------------------
        # D) Total
        # -------------------------
        total_loss = (
            reconstruction_loss
            + global_kl_beta * global_kl_loss
            + local_kl_beta * local_kl_loss
        )

        total_kl_unweighted = global_kl_loss + local_kl_loss
        kl_ratio = (reconstruction_loss / (total_kl_unweighted + 1e-8)).detach()

        return (
            total_loss,
            reconstruction_loss,   # avg over batch (sum tokens per ex, then mean)
            global_kl_loss,        # sum dims per ex, mean over batch (charged)
            local_kl_loss,         # sum dims+sentences per ex, mean over batch (charged)
            kl_ratio,
            mean_token_loss,       # avg per token (for logging)
            global_kl_raw_mean,    # sum dims per ex, mean over batch (raw)
            local_kl_raw_mean,     # sum dims+sentences per ex, mean over batch (raw)
            global_frac_dims_clamped,
            local_frac_dims_clamped,
        )


In [13]:
def save_checkpoint(
    inference_net,
    generative_net,
    optimizer,
    epoch,
    val_loss,
    scheduler=None,
    is_best=False,
    filename="hvae_checkpoint.pth",
):
    """
    Saves model state to a local file and uploads it to W&B as an artifact.
    - Writes: {filename}_epoch_{epoch}
    - Uploads that exact file (so W&B doesn't error).
    - Aliases: always "latest", plus "best" if is_best=True
    """
    # 1) Build state dict
    state = {
        "epoch": epoch,
        "inference_state_dict": inference_net.state_dict(),
        "generative_state_dict": generative_net.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "val_loss": float(val_loss) if hasattr(val_loss, "__float__") else val_loss,
    }

    # 2) Save locally
    save_path = f"{filename}_epoch_{epoch}"
    torch.save(state, save_path)

    # Sanity check (gives a clearer error than wandb's)
    if not os.path.isfile(save_path):
        raise FileNotFoundError(f"Checkpoint was not written: {save_path!r}")

    # 3) Create artifact (use a stable name so versions accumulate)
    artifact = wandb.Artifact(
        name="hvae-model",          # stable artifact name
        type="model",
        metadata={"epoch": epoch, "val_loss": state["val_loss"]},
    )

    # 4) Add the saved file
    # Store it in the artifact with a clean, consistent logical name
    artifact.add_file(save_path, name=filename)

    # 5) Log with aliases
    aliases = ["latest"]
    if is_best:
        aliases.append("best")

    wandb.log_artifact(artifact, aliases=aliases)

    # (Optional) return the path for convenience
    return save_path

In [14]:
# Hyper params dict to initialize the models
hyperParams = {
    # --- Architecture Constraints ---
    'gpt2_model_name': 'gpt2',   # Start with 'gpt2' (124M params). Use 'gpt2-medium' only if you have high VRAM.
    'd_model': 768,              # MUST be 768 for 'gpt2', 1024 for 'gpt2-medium', 1280 for 'gpt2-large'.
    'vocab_size': 50257,         # Standard GPT-2 vocabulary size.

    # --- Latent Space ---
    'latent_dim': 64,           # 32-128 is standard. Too large = posterior collapse; Too small = poor reconstruction.

    # --- Data Dimensions (Abstract specific) ---
    'max_sentences': 10,         # Average abstract is 5-8 sentences; 10 provides a safety buffer.
    'max_words': 50,             # Average sentence is 20-30 words; 50 covers outliers.

    # --- Inference Network (Encoder) ---
    'encoder_layers': 2,         # Keep encoder shallow (2-4) so the heavy lifting happens in the decoder/latent space.
    'encoder_heads': 8,          # Standard for d_model=768 (768 / 8 = 96 dim per head).
    'encoder_dropout': 0.1,      # Standard transformer dropout.

    # --- Sentence Decoder (GRU) ---
    'gru_layers': 1,             # 1 layer is sufficient for the high-level plan; more layers add unnecessary complexity.

    # --- Special Tokens ---
    'pad_index': 50256,          # GPT-2 uses EOS (50256) as PAD by default unless you added a new token.
    'word_dropout_rate' : 0,
    'plan_dropout_rate':0,
    'mask_token_id':10
}

In [15]:
# helper function to tokezine text with bert and gpt2
def process_dual_stream(text, enc_tokenizer, dec_tokenizer, max_sentences, max_words):
    """
    Parses text into TWO sets of tensors (encoder + decoder), shaped as:
      [max_sentences][max_words]

    Encoder (DistilBERT):
      [CLS] sent_tokens [SEP] + pad

    Decoder (GPT-2):
      For each sentence:
        [BOS] sent_tokens [<EOS>]                (sentence terminator)
      For the VERY LAST sentence:
        [BOS] sent_tokens [<EOS>] [<|endoftext|>] (document terminator)

    Assumes:
      - Raw text contains "<BOS>" markers before each sentence and "<EOS>" delimiters
      - Raw text ends with "<|endoftext|>" (may appear once at end)
      - dec_tokenizer has bos_token="<BOS>", pad_token="<PAD>", and "<EOS>" added
    """

    # ----------------------------
    # 0) Resolve special token IDs
    # ----------------------------
    bos_id = dec_tokenizer.bos_token_id              # should be ID of "<BOS>"
    pad_id_dec = dec_tokenizer.pad_token_id          # should be ID of "<PAD>"
    eot_id = dec_tokenizer.eos_token_id              # GPT-2 original "<|endoftext|>"

    eos_sent_id = dec_tokenizer.convert_tokens_to_ids("<EOS>")  # your sentence EOS
    if eos_sent_id is None:
        raise ValueError('Decoder tokenizer does not know "<EOS>" (did you add_special_tokens?).')

    if bos_id is None or pad_id_dec is None or eot_id is None:
        raise ValueError("Decoder tokenizer is missing bos/pad/eos token ids. Check tokenizer setup.")

    # ----------------------------
    # 1) Clean + split into sentences
    # ----------------------------
    # Remove any explicit document terminator strings from the raw text;
    # we will add the doc terminator as an ID ourselves at the end.
    text = text.replace("<|endoftext|>", "").strip()

    raw_sentences = text.split("<EOS>")
    sentences = []
    for s in raw_sentences:
        s = s.strip()
        if not s:
            continue
        # Raw data prefixes each sentence with "<BOS>" as text; strip it
        if s.startswith("<BOS>"):
            s = s[len("<BOS>"):].strip()
        # In case of accidental repetition, strip repeatedly
        while s.startswith("<BOS>"):
            s = s[len("<BOS>"):].strip()
        if s:
            sentences.append(s)

    # ----------------------------
    # 2) Tokenize each sentence into rows
    # ----------------------------
    enc_ids_rows, enc_mask_rows = [], []
    dec_ids_rows, dec_mask_rows = [], []

    num_sentences = len(sentences)

    for i, sent in enumerate(sentences):
        # --- A) ENCODER (DistilBERT) ---
        enc_tokens = enc_tokenizer.encode(sent, add_special_tokens=False)
        enc_tokens = enc_tokens[: max_words - 2]  # reserve [CLS],[SEP]
        full_enc = [enc_tokenizer.cls_token_id] + enc_tokens + [enc_tokenizer.sep_token_id]

        enc_mask = [1] * len(full_enc)
        pad_len_enc = max_words - len(full_enc)
        if pad_len_enc > 0:
            full_enc += [enc_tokenizer.pad_token_id] * pad_len_enc
            enc_mask += [0] * pad_len_enc

        enc_ids_rows.append(full_enc)
        enc_mask_rows.append(enc_mask)

        # --- B) DECODER (GPT-2) ---
        dec_tokens = dec_tokenizer.encode(sent, add_special_tokens=False)

        is_last_sentence = (i == num_sentences - 1)

        if is_last_sentence:
            # Reserve 3 spots: BOS, <EOS>, <|endoftext|>
            dec_tokens = dec_tokens[: max_words - 3]
            full_dec = [bos_id] + dec_tokens + [eos_sent_id] + [eot_id]
        else:
            # Reserve 2 spots: BOS, <EOS>
            dec_tokens = dec_tokens[: max_words - 2]
            full_dec = [bos_id] + dec_tokens + [eos_sent_id]

        dec_mask = [1] * len(full_dec)
        pad_len_dec = max_words - len(full_dec)
        if pad_len_dec > 0:
            full_dec += [pad_id_dec] * pad_len_dec
            dec_mask += [0] * pad_len_dec

        dec_ids_rows.append(full_dec)
        dec_mask_rows.append(dec_mask)

    # ----------------------------
    # 3) Vertical padding to max_sentences
    # ----------------------------
    while len(enc_ids_rows) < max_sentences:
        enc_ids_rows.append([enc_tokenizer.pad_token_id] * max_words)
        enc_mask_rows.append([0] * max_words)

        dec_ids_rows.append([pad_id_dec] * max_words)
        dec_mask_rows.append([0] * max_words)

    # Truncate vertically if too many sentences
    enc_ids_rows = enc_ids_rows[:max_sentences]
    enc_mask_rows = enc_mask_rows[:max_sentences]
    dec_ids_rows = dec_ids_rows[:max_sentences]
    dec_mask_rows = dec_mask_rows[:max_sentences]

    return enc_ids_rows, enc_mask_rows, dec_ids_rows, dec_mask_rows


In [16]:
# Dataset class created here
class HVAEDataset(Dataset):
    def __init__(self, dataframe, enc_tokenizer, dec_tokenizer, max_sentences, max_words):
        self.data = dataframe['structured_abstract'].tolist()
        self.enc_tokenizer = enc_tokenizer # DistilBERT
        self.dec_tokenizer = dec_tokenizer # GPT-2
        self.max_sentences = max_sentences
        self.max_words = max_words

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]

        enc_ids, enc_mask, dec_ids, dec_mask = process_dual_stream(
            text,
            self.enc_tokenizer,
            self.dec_tokenizer,
            self.max_sentences,
            self.max_words,
        )

        return {
            'enc_input_ids': torch.tensor(enc_ids, dtype=torch.long),
            'enc_word_mask': torch.tensor(enc_mask, dtype=torch.long), # DistilBERT mask
            'dec_input_ids': torch.tensor(dec_ids, dtype=torch.long),
            'dec_word_mask': torch.tensor(dec_mask, dtype=torch.long)  # GPT-2 mask
        }

def create_dataloaders(df, enc_tokenizer,dec_tokenizer, hyperparams, batch_size=32):
    # 1. Create Dataset
    dataset = HVAEDataset(
        dataframe=df,
        enc_tokenizer=enc_tokenizer,
        dec_tokenizer = dec_tokenizer,
        max_sentences=hyperparams['max_sentences'],
        max_words=hyperparams['max_words'],
    )

    # 2. Create DataLoader
    # num_workers=0 is safer for debugging; increase for speed later
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        drop_last=True
    )

    return dataloader

In [17]:
# Starting wandb project
wandb.init(
    project="HVAE-kv_injection_full_data",
    config=hyperParams,
    name = 'only_plan_10epochs'
)

In [18]:
# function created for cyclic beta annealing
def get_kl_beta(
    epoch_index,
    batch_idx,
    steps_per_epoch,
    MIN_BETA=0.001,
    MAX_BETA=0.1,
    CYCLE_EPOCHS=20,
    MIN_HOLD_FRAC=0.2,
    RAMP_FRAC=0.5
):
    """
    Cyclical Beta Schedule:
    [MIN_BETA ..... / RAMP / ..... MAX_BETA .....] -> Repeat
    """
    total_steps_in_cycle = steps_per_epoch * CYCLE_EPOCHS
    current_global_step = epoch_index * steps_per_epoch + batch_idx

    cycle_progress = (current_global_step % total_steps_in_cycle) / total_steps_in_cycle

    phase_a_end = MIN_HOLD_FRAC
    phase_b_end = MIN_HOLD_FRAC + RAMP_FRAC

    if cycle_progress < phase_a_end:
        current_beta = MIN_BETA

    elif cycle_progress < phase_b_end:
        ramp_progress = (cycle_progress - phase_a_end) / RAMP_FRAC
        current_beta = MIN_BETA + ramp_progress * (MAX_BETA - MIN_BETA)

    else:
        current_beta = MAX_BETA

    return (current_beta, current_beta)

In [19]:
def compute_active_units(mu_global, mu_local, threshold=0.01):
    """
    mu_global: (B, D)
    mu_local:  (B, S, D)
    Returns metrics dict (includes a wandb.Image heatmap).
    """
    assert mu_global.dim() == 2, f"mu_global should be (B,D), got {mu_global.shape}"
    assert mu_local.dim() == 3, f"mu_local should be (B,S,D), got {mu_local.shape}"
    B, S, D = mu_local.shape
    assert mu_global.shape == (B, D), f"Expected mu_global {(B,D)}, got {mu_global.shape}"

    # Detach so logging doesn't keep graphs around
    mu_g = mu_global.detach()
    mu_l = mu_local.detach()

    # 1) Global AU
    global_vars = mu_g.var(dim=0, unbiased=False)          # (D,)
    num_active_global = (global_vars > threshold).sum().item()

    # 2) Local AU (aggregate across all sentences)
    local_flat = mu_l.flatten(0, 1)                        # (B*S, D)
    local_vars_agg = local_flat.var(dim=0, unbiased=False) # (D,)
    num_active_local = (local_vars_agg > threshold).sum().item()

    # 3) Local activity map per sentence position
    local_activity_map = mu_l.var(dim=0, unbiased=False)   # (S, D)

    metrics = {
        "AU/Global_Count": num_active_global,
        "AU/Local_Count": num_active_local,
        "AU/Global_Variance_Avg": global_vars.mean().item(),
        "AU/Local_Variance_Avg": local_vars_agg.mean().item(),
    }

    # Optional: per-sentence active counts (often useful)
    per_sent_active = (local_activity_map > threshold).sum(dim=1)  # (S,)
    metrics["AU/Local_Count_PerSentence_Mean"] = per_sent_active.float().mean().item()

    # Visualization (binary mask)
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap((local_activity_map.cpu().numpy() > threshold), ax=ax, cbar=False)
    ax.set_title("Active Units per Sentence Position (Binary)")
    ax.set_ylabel("Sentence Index")
    ax.set_xlabel("Latent Dimension")

    metrics["AU/Local_Heatmap"] = wandb.Image(fig)
    plt.close(fig)

    return metrics


In [20]:
# loading a saved checkpoint from wandb
def load_checkpoint_from_wandb(
    artifact_path,
    inference_net,
    generative_net,
    optimizer,
    scheduler=None,
    filename="hvae_checkpoint.pth"
):
    """
    Downloads a specific artifact from WandB, loads the state dictionaries,
    and returns the epoch to resume from.
    """
    print(f"Resuming from WandB artifact: {artifact_path}")

    # 1. Download the artifact
    # explicit run init is usually required if not already active,
    # but wandb.use_artifact works if run is active.
    artifact = wandb.use_artifact(artifact_path, type='model')
    artifact_dir = artifact.download()
    filepath = os.path.join(artifact_dir, filename)

    # 2. Load the file
    if torch.cuda.is_available():
        checkpoint = torch.load(filepath)
    else:
        checkpoint = torch.load(filepath, map_location=torch.device('cpu'))

    # 3. Restore states
    inference_net.load_state_dict(checkpoint['inference_state_dict'])
    generative_net.load_state_dict(checkpoint['generative_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if scheduler and checkpoint.get('scheduler_state_dict'):
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # 4. Return the next epoch
    # If we saved at epoch 9 (finished), we want to start at 10.
    start_epoch = checkpoint['epoch'] + 1
    val_loss = checkpoint.get('val_loss', 'N/A')

    print(f"Checkpoint loaded. Resuming from Epoch {start_epoch} (Last Val Loss: {val_loss})")
    return start_epoch

In [21]:
#  train one epoch
def reparameterize(mu, sigma2, eps=1e-8):
    sigma2 = torch.clamp(sigma2, min=eps)
    std = torch.sqrt(sigma2)
    return mu + torch.randn_like(std) * std

def train_one_epoch(
    inference_net,
    generative_net,
    loss_module,
    dataloader,
    optimizer,
    device,
    epoch_index,
    total_epochs,
    au_batch,
    scheduler = None
):
    inference_net.train()
    generative_net.train()

    total_epoch_loss = 0
    total_recon_loss = 0
    total_kl_loss = 0

    local_b = 0.0
    if epoch_index >=4:
        local_b = 0.02


    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch_index+1}")
    steps_per_epoch = len(dataloader)

    target_batch_size = 64
    batch_size = dataloader.batch_size # e.g., 8
    accumulation_steps = target_batch_size // batch_size

    optimizer.zero_grad()

    if epoch_index == 0:
         for param in generative_net.gpt2_model.parameters():
            param.requires_grad = True



    for batch_idx, batch in enumerate(progress_bar):

        global_beta,local_beta = get_kl_beta(
                    epoch_index,
                    batch_idx,
                    steps_per_epoch,
                    MIN_BETA=0,  # Start very low to encourage encoding early
                    MAX_BETA=0.5,    # Cap at 0.1 to avoid overpowering the reconstruction
                    CYCLE_EPOCHS=1, # Length of one full cycle
                    MIN_HOLD_FRAC=0.5, # 20% of cycle spent holding MIN_BETA
                    RAMP_FRAC=0.25      # 50% of cycle spent ramping up
      )
        # 1. Prepare Data
        # Assuming batch is a tuple/list; adjust unpacking based on your CollateFn
        enc_input_ids = batch['enc_input_ids'].to(device)
        enc_word_mask = batch['enc_word_mask'].to(device)
        dec_input_ids = batch['dec_input_ids'].to(device)
        dec_word_mask = batch['dec_word_mask'].to(device)

        # Target IDs are typically the same as input_ids for reconstruction
        # The Loss module should handle shifting (input[t] -> target[t+1]) internally
        # or via masking.
        target_ids = dec_input_ids.clone()


        # 2. Inference Network (Encoder)
        # Forward pass to get posterior parameters q(z|x)
        mu_t_q, sigma2_t_q, mu_i_q, sigma2_i_q = inference_net(dec_input_ids, dec_word_mask)

        # 3. Sampling (Reparameterization)
        z_t = reparameterize(mu_t_q, sigma2_t_q)
        z_i_samples = reparameterize(mu_i_q, sigma2_i_q)

        # 4. Generative Network (Decoder)
        # Reconstruct inputs based on samples and calculate priors p(z)
        reconstruction_logits, mu_i_p, sigma2_i_p = generative_net(
            dec_input_ids,
            dec_word_mask,
            z_t,
            z_i_samples
        )


        # 5. Loss Calculation
        loss, recon, global_kl, local_kl, kl_ratio, per_token_loss, global_klraw, local_raw, global_clamp_frac, local_clamp_frac = loss_module(
            # From Encoder (Posterior q)
            mu_t_q, sigma2_t_q, mu_i_q, sigma2_i_q,
            # From Decoder (Likelihood + Prior p)
            reconstruction_logits, mu_i_p, sigma2_i_p,
            # Targets
            target_ids, dec_word_mask,
            # Annealing
            local_kl_beta=global_beta,
            global_kl_beta = global_beta,

        )

        scaled_loss = loss / accumulation_steps
        scaled_loss.backward()
        if batch_idx == 0 and epoch_index == 0:
          progress_bar.write(f"Global Mu Mean:   {mu_t_q.mean().item():.5f} | Std: {mu_t_q.std().item():.5f}")
          progress_bar.write(f"Global Sigma2 Mean: {sigma2_t_q.mean().item():.5f}")

        if (batch_idx + 1) % accumulation_steps == 0:
            # Gradient Clipping
            utils.clip_grad_norm_(inference_net.parameters(), max_norm=1.0)
            utils.clip_grad_norm_(generative_net.parameters(), max_norm=1.0)

            optimizer.step()
            if scheduler is not None:
                scheduler.step()

            optimizer.zero_grad() # Reset for next set of accumulation



        current_lr_1 = optimizer.param_groups[0]['lr']
        current_lr_2 = optimizer.param_groups[1]['lr']

        # 7. Logging
        total_epoch_loss += loss.item()
        total_recon_loss += recon.item()
        # Sum global and local KL for display
        total_kl_loss += (global_kl.item() + local_kl.item())
        current_kl = global_kl.item() + local_kl.item()
        if batch_idx % 10000 == 0:



            log_dict = {
                "Losses/loss": loss.item(),
                "Losses/recon": recon.item(),
                "KL/kl_ratio": kl_ratio.item(),
                "KL/kl_local": local_kl.item(),
                "KL/kl_total": current_kl,
                "KL/kl_global": global_kl.item(),
                "KL/kl_global_raw": global_klraw.item(),
                "KL/kl_local_raw": local_raw.item(),
                "KL/kl_global_clamp_frac": global_clamp_frac.item(),
                "KL/kl_local_clamp_frac": local_clamp_frac.item(),
                "Losses/per_token_loss": per_token_loss.item(),
                'Betas/local_kl_beta':global_beta,
                'Betas/global_kl_beta': global_beta,
                'LRs/lr1': current_lr_1,
                'LRs/lr2': current_lr_2,
                }



            au_mu_t_q_cpu, au_mu_i_q_cpu = encode_au_batch_in_chunks(
                    au_batch,
                    inference_net,
                    device=device,
                    chunk_size=16,     # start small; increase until it fits
                    use_amp=False      # optional, often helps memory
                )
            au_metrics = compute_active_units(
                    au_mu_t_q_cpu,
                    au_mu_i_q_cpu,
                    threshold=0.01
                    )

            log_dict.update(au_metrics)

            wandb.log(log_dict)

    # save_checkpoint(
    #     inference_net=inference_net,
    #     generative_net=generative_net,
    #     optimizer=optimizer,
    #     epoch=epoch,
    #     val_loss=0.1,
    #     scheduler=scheduler,
    #     is_best=False
    # )

In [22]:
# adding the special tokens to the tokenizer
# 1. Load standard tokenizers
dec_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
enc_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# 2. Add Special Tokens
# We use 'additional_special_tokens' for custom tokens like EOT
special_tokens_dict = {
    "bos_token": "<BOS>",
    "pad_token": "<PAD>",

    "additional_special_tokens": [
        "<EOS>",
        "<MASK>",
    ],
}

dec_tokenizer.add_special_tokens(special_tokens_dict)
# 3. Get IDs
# Standard attributes exist for bos/eos/pad/mask
pad_idx = dec_tokenizer.pad_token_id                 # <PAD>
bos_idx = dec_tokenizer.bos_token_id                 # <BOS>

# GPT-2's original eos_token (<|endoftext|>)
gpt2_eos_id = dec_tokenizer.eos_token_id

# Your custom tokens
eos_idx = dec_tokenizer.convert_tokens_to_ids("<EOS>")
mask_idx = dec_tokenizer.convert_tokens_to_ids("<MASK>")

new_vocab_size = len(dec_tokenizer)

print(f"New Vocab Size: {new_vocab_size}")
print(f"PAD ID: {pad_idx} | BOS ID: {bos_idx}")
print(f"GPT2 EOS ID (<|endoftext|>): {gpt2_eos_id}")
print(f"Custom EOS ID (<EOS>): {eos_idx}")
print(f"MASK ID: {mask_idx}")

# 3. Update hyperparameters
hyperParams['vocab_size'] = new_vocab_size
hyperParams['pad_index'] = pad_idx
hyperParams['bos_index'] = bos_idx
hyperParams['gpt2_eos_id'] = gpt2_eos_id   # optional, but nice to keep
hyperParams['eos_token_id'] = eos_idx      # your custom EOS
hyperParams['mask_token_id'] = mask_idx

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


New Vocab Size: 50261
PAD ID: 50258 | BOS ID: 50257
GPT2 EOS ID (<|endoftext|>): 50256
Custom EOS ID (<EOS>): 50259
MASK ID: 50260


In [23]:
# Creating dataloaders
train_subset = train_df.sample(frac=1, random_state=42)
test_subset = valid_df.sample(frac=0.01, random_state=42)
train_loader = create_dataloaders(train_subset, enc_tokenizer,dec_tokenizer, hyperParams, batch_size=32)
valid_loader = create_dataloaders(test_subset, enc_tokenizer,dec_tokenizer ,hyperParams, batch_size=8)

In [24]:
@contextmanager
def _eval_mode(model):
    was_training = model.training
    model.eval()
    try:
        yield
    finally:
        model.train(was_training)

@torch.no_grad()
def encode_au_batch_in_chunks(
    au_batch: dict,
    inference_net,
    device,
    chunk_size: int = 8,
    use_amp: bool = False,
    ids_key: str = "dec_input_ids",
    mask_key: str = "dec_word_mask",
):
    """
    Takes a *big* CPU batch (au_batch) and runs inference_net on GPU in chunks.
    Returns:
      mu_t_all_cpu: (N, D)
      mu_i_all_cpu: (N, S, D)
    """
    ids = au_batch[ids_key]   # keep on CPU
    mask = au_batch[mask_key] # keep on CPU
    N = ids.shape[0]

    mu_t_list = []
    mu_i_list = []

    amp_enabled = use_amp and (device.type == "cuda")

    with _eval_mode(inference_net):
        for start in range(0, N, chunk_size):
            end = min(N, start + chunk_size)

            ids_chunk  = ids[start:end].to(device, non_blocking=True)
            mask_chunk = mask[start:end].to(device, non_blocking=True)

            if amp_enabled:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    mu_t_q, _, mu_i_q, _ = inference_net(ids_chunk, mask_chunk)
            else:
                mu_t_q, _, mu_i_q, _ = inference_net(ids_chunk, mask_chunk)

            mu_t_list.append(mu_t_q.detach().cpu())
            mu_i_list.append(mu_i_q.detach().cpu())

            # free chunk GPU tensors ASAP
            del ids_chunk, mask_chunk, mu_t_q, mu_i_q

    mu_t_all_cpu = torch.cat(mu_t_list, dim=0)
    mu_i_all_cpu = torch.cat(mu_i_list, dim=0)
    return mu_t_all_cpu, mu_i_all_cpu

In [25]:
def make_big_batch(loader, num_minibatches=50, device=None):
    """
    Concatenate `num_minibatches` minibatches from `loader` into one big batch.

    - Works for batches that are dict[str, Tensor] (most common).
    - If a value is not a Tensor, it will be collected into a list.
    - If `device` is provided, moves tensors onto that device.

    Returns:
      big_batch: dict with same keys as minibatch
    """
    it = iter(loader)
    batch_list = []

    for _ in range(num_minibatches):
        b = next(it)  # StopIteration if loader is too short

        if device is not None:
            b = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}

        batch_list.append(b)

    # concatenate
    big = {}
    keys = batch_list[0].keys()

    for k in keys:
        v0 = batch_list[0][k]
        if torch.is_tensor(v0):
            # concat along batch dimension
            big[k] = torch.cat([b[k] for b in batch_list], dim=0)
        else:
            big[k] = [b[k] for b in batch_list]

    return big


In [26]:
au_batch = make_big_batch(train_loader, num_minibatches=50)
au_batch['dec_input_ids'].shape

torch.Size([1600, 10, 50])

In [27]:
# Models initialization + training is done here
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

WANDB_ARTIFACT_PATH = "yasir-alam14/HVAE-kv_injection_full_data/hvae-model:latest" # Set this to None if starting fresh
RESUME_TRAINING = True

num_epochs = 10
# 1. Calculate total training steps
target_batch_size = 64
batch_size = train_loader.batch_size # e.g., 8
accumulation_steps = target_batch_size // batch_size

total_steps = (num_epochs * len(train_loader)) // accumulation_steps

# 2. Define Warmup (e.g., 5% of total steps)
warmup_steps = int(0.1 * total_steps)
decay_steps = total_steps - warmup_steps

print(f"Total Steps: {total_steps} | Warmup Steps: {warmup_steps}")

# Instantiate Models
inference_net = HT_HVAE_InferenceNetwork(hyperParams).to(device)
generative_net = HT_HVAE_GenerativeNetwork(hyperParams).to(device)
loss_module = HT_HVAE_Loss(hyperParams).to(device)


# Define your Learning Rates
LR_GPT2 = 5e-5   # Low LR for pre-trained weights
LR_BERT = 5e-5
LR_REST = 3e-4   # High LR for new Encoders/GRU/MLPs

gpt2_params = list(generative_net.gpt2_model.parameters())
distilbert_params = list(inference_net.word_encoder.parameters())

# 2. Identify "Everything Else"
# Create a set of IDs for parameters already assigned to avoid duplication
assigned_ids = set(map(id, gpt2_params + distilbert_params))
scratch_params = []

# Iterate through both networks and collect unassigned parameters
for model in [inference_net, generative_net]:
    for param in model.parameters():
        if id(param) not in assigned_ids:
            scratch_params.append(param)

# 3. Create 3-Group Optimizer
optimizer = torch.optim.AdamW([
    # No weight decay for pre-trained models to preserve embedding norms
    {'params': gpt2_params, 'lr': LR_GPT2, 'weight_decay': 0.0},
    {'params': distilbert_params, 'lr': LR_BERT, 'weight_decay': 0.0},

    # Keep weight decay for scratch parameters to prevent them from exploding
    {'params': scratch_params, 'lr': LR_REST, 'weight_decay': 0.00}
])


scheduler_warmup = LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=warmup_steps
)


scheduler_decay = CosineAnnealingLR(
    optimizer,
    T_max=decay_steps,
    eta_min=1e-6
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[scheduler_warmup, scheduler_decay],
    milestones=[warmup_steps]
)

start_epoch = 0

if RESUME_TRAINING and WANDB_ARTIFACT_PATH:
    if wandb.run is None:
        wandb.init(project="my_project", resume="allow")

    start_epoch = load_checkpoint_from_wandb(
        artifact_path=WANDB_ARTIFACT_PATH,
        inference_net=inference_net,
        generative_net=generative_net,
        optimizer=optimizer,
        scheduler=scheduler
    )


special_tokens_set = {'<PAD>', '<BOS>', '<EOS>', '<|endoftext|>'}

for epoch in range(start_epoch, num_epochs):
    if epoch >= num_epochs:
        print("Training already completed for this number of epochs.")
        break
    train_one_epoch(inference_net, generative_net, loss_module, train_loader, optimizer, device, epoch, num_epochs, au_batch,scheduler)
    #results = evaluate_model(inference_net, generative_net, loss_module, valid_loader, tokenizer, device, special_tokens_set)
    #validation_logs = {f"validation/{k}": v for k, v in results.items()}
    #wandb.log(validation_logs)
    if (epoch + 1 )%1 == 0:
        save_checkpoint(
            inference_net=inference_net,
            generative_net=generative_net,
            optimizer=optimizer,
            epoch=epoch,
            val_loss=0.1,
            scheduler=scheduler,
            is_best=False
        )
    break

Total Steps: 62070 | Warmup Steps: 6207


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Resuming from WandB artifact: yasir-alam14/HVAE-kv_injection_full_data/hvae-model:latest


wandb: Downloading large artifact 'hvae-model:latest', 5962.91MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:01.8 (3259.6MB/s)


Checkpoint loaded. Resuming from Epoch 4 (Last Val Loss: 0.1)


Epoch 5:   0%|          | 24/12414 [00:32<4:42:21,  1.37s/it]


KeyboardInterrupt: 

In [ ]:
# checking for nans in the model weights
for name, param in inference_net.named_parameters():
    if torch.isnan(param).any():
        print(f"FATAL: Found NaN in model weights: {name}")
        break
else:
    print("Weights are clean. The issue is in the forward pass calculation.")

In [ ]:
jfkuytdkjy,gflckhgcdxytx

In [ ]:
# Testing starts here

In [ ]:
# making the encoder decoder to eval to have inference settings
generative_net.eval()
inference_net.eval()
x =1

In [ ]:
# first batch sampled from the training data do perform certain tests on it
first_batch = next(iter(train_loader))

In [ ]:
# We calculate the losses for the first batch
# enc_input_ids = first_batch['enc_input_ids'].to(device)
# enc_word_mask = first_batch['enc_word_mask'].to(device)
# dec_input_ids = first_batch['dec_input_ids'].to(device)
# dec_word_mask = first_batch['dec_word_mask'].to(device)

# # Target IDs are typically the same as input_ids for reconstruction
# # The Loss module should handle shifting (input[t] -> target[t+1]) internally
# # or via masking.
# target_ids = dec_input_ids.clone()


# # 2. Inference Network (Encoder)
# # Forward pass to get posterior parameters q(z|x)
# mu_t_q, sigma2_t_q, mu_i_q, sigma2_i_q = inference_net(dec_input_ids, dec_word_mask)

# # 3. Sampling (Reparameterization)
# z_t = reparameterize(mu_t_q, sigma2_t_q)
# z_i_samples = reparameterize(mu_i_q, sigma2_i_q)

# # 4. Generative Network (Decoder)
# # Reconstruct inputs based on samples and calculate priors p(z)
# reconstruction_logits, mu_i_p, sigma2_i_p = generative_net(
#     dec_input_ids,
#     dec_word_mask,
#     z_t,
#     z_i_samples
# )

# # 5. Loss Calculation
# loss, recon, global_kl, local_kl, kl_ratio, per_token_loss = loss_module(
#     # From Encoder (Posterior q)
#     mu_t_q, sigma2_t_q, mu_i_q, sigma2_i_q,
#     # From Decoder (Likelihood + Prior p)
#     reconstruction_logits, mu_i_p, sigma2_i_p,
#     # Targets
#     target_ids, dec_word_mask,
#     # Annealing
#     local_kl_beta=1,
#     global_kl_beta = 1,

# )

In [ ]:
# here we want to confirm that the zi_samples we got are the right shape
# z_i_samples.shape

In [ ]:
# printing the losses
# print(f"Recon loss is {per_token_loss}\n global_kl is {global_kl} \n local is {local_kl}")

In [ ]:
# What is the perplexity
# np.exp(per_token_loss.detach().cpu())

In [ ]:
# Losses but with shuffled zt
# enc_input_ids = first_batch['enc_input_ids'].to(device)
# enc_word_mask = first_batch['enc_word_mask'].to(device)
# dec_input_ids = first_batch['dec_input_ids'].to(device)
# dec_word_mask = first_batch['dec_word_mask'].to(device)

# # Target IDs are typically the same as input_ids for reconstruction
# # The Loss module should handle shifting (input[t] -> target[t+1]) internally
# # or via masking.
# target_ids = dec_input_ids.clone()


# # 2. Inference Network (Encoder)
# # Forward pass to get posterior parameters q(z|x)
# mu_t_q, sigma2_t_q, mu_i_q, sigma2_i_q = inference_net(dec_input_ids, dec_word_mask)

# # 3. Sampling (Reparameterization)
# z_t = reparameterize(mu_t_q, sigma2_t_q)
# z_i_samples = reparameterize(mu_i_q, sigma2_i_q)

# perm = torch.randperm(z_t.size(0))
# z_t_permed, z_i_samples = z_t[perm], z_i_samples

# # 4. Generative Network (Decoder)
# # Reconstruct inputs based on samples and calculate priors p(z)
# reconstruction_logits, mu_i_p, sigma2_i_p = generative_net(
#     dec_input_ids,
#     dec_word_mask,
#     z_t_permed,
#     z_i_samples
# )

# # 5. Loss Calculation
# loss, recon, global_kl, local_kl, kl_ratio, per_token_loss = loss_module(
#     # From Encoder (Posterior q)
#     mu_t_q, sigma2_t_q, mu_i_q, sigma2_i_q,
#     # From Decoder (Likelihood + Prior p)
#     reconstruction_logits, mu_i_p, sigma2_i_p,
#     # Targets
#     target_ids, dec_word_mask,
#     # Annealing
#     local_kl_beta=1,
#     global_kl_beta = 1,

# )

In [ ]:
# print(f"Recon loss is {per_token_loss}\n global_kl is {global_kl} \n local is {local_kl}")

In [ ]:
# np.exp(per_token_loss.detach().cpu())

In [ ]:
# Losses with zi shuffled
# enc_input_ids = first_batch['enc_input_ids'].to(device)
# enc_word_mask = first_batch['enc_word_mask'].to(device)
# dec_input_ids = first_batch['dec_input_ids'].to(device)
# dec_word_mask = first_batch['dec_word_mask'].to(device)

# # Target IDs are typically the same as input_ids for reconstruction
# # The Loss module should handle shifting (input[t] -> target[t+1]) internally
# # or via masking.
# target_ids = dec_input_ids.clone()


# # 2. Inference Network (Encoder)
# # Forward pass to get posterior parameters q(z|x)
# mu_t_q, sigma2_t_q, mu_i_q, sigma2_i_q = inference_net(dec_input_ids, dec_word_mask)

# # 3. Sampling (Reparameterization)
# z_t = reparameterize(mu_t_q, sigma2_t_q)
# z_i_samples = reparameterize(mu_i_q, sigma2_i_q)
# perm = torch.randperm(z_t.size(0))
# z_t, z_i_samples_permed = z_t, z_i_samples[perm]

# # 4. Generative Network (Decoder)
# # Reconstruct inputs based on samples and calculate priors p(z)
# reconstruction_logits, mu_i_p, sigma2_i_p = generative_net(
#     dec_input_ids,
#     dec_word_mask,
#     z_t,
#     z_i_samples_permed
# )

# # 5. Loss Calculation
# loss, recon, global_kl, local_kl, kl_ratio, per_token_loss = loss_module(
#     # From Encoder (Posterior q)
#     mu_t_q, sigma2_t_q, mu_i_q, sigma2_i_q,
#     # From Decoder (Likelihood + Prior p)
#     reconstruction_logits, mu_i_p, sigma2_i_p,
#     # Targets
#     target_ids, dec_word_mask,
#     # Annealing
#     local_kl_beta=1,
#     global_kl_beta = 1,

# )

In [ ]:
# print(f"Recon loss is {per_token_loss}\n global_kl is {global_kl} \n local is {local_kl}")

In [ ]:
# np.exp(per_token_loss.detach().cpu())

In [ ]:
# mu_i_q.shape

In [ ]:
# generating the masks and latents to do testings if the code for losses was not executed
enc_input_ids = first_batch['enc_input_ids'].to(device)
enc_word_mask = first_batch['enc_word_mask'].to(device)
dec_input_ids = first_batch['dec_input_ids'].to(device)
dec_word_mask = first_batch['dec_word_mask'].to(device)

# Target IDs are typically the same as input_ids for reconstruction
# The Loss module should handle shifting (input[t] -> target[t+1]) internally
# or via masking.
target_ids = dec_input_ids.clone()


# 2. Inference Network (Encoder)
# Forward pass to get posterior parameters q(z|x)
mu_t_q, sigma2_t_q, mu_i_q, sigma2_i_q = inference_net(dec_input_ids, dec_word_mask)

# 3. Sampling (Reparameterization)
z_t = reparameterize(mu_t_q, sigma2_t_q)
z_i_samples = reparameterize(mu_i_q, sigma2_i_q)

In [ ]:
au_metrics = compute_active_units(
            mu_t_q.detach().cpu(),
          mu_i_q.detach().cpu(),
          threshold=0.01
        )

In [ ]:
print(au_metrics)

In [ ]:
# latents so that we can generate the text from them
test_zt = mu_t_q[0]
test_zi = mu_i_q[0]
test_zt = test_zt.unsqueeze(0)
test_zi = test_zi.unsqueeze(0)
real_ids = first_batch['dec_input_ids'][0].unsqueeze(0).to(device)

In [ ]:
# function to decode the generated ids to text
def decode_generated_sequences(generated_sequences, tokenizer, skip_special_tokens=True):
    """
    Decodes the list of tensors returned by the inference method into text.

    Args:
        generated_sequences (list[torch.Tensor]): List of tensors where each tensor
                                                  has shape (Batch, Seq_Len).
        tokenizer: A tokenizer with a .decode() method (e.g., HuggingFace).
        skip_special_tokens (bool): If True, removes BOS/EOS/PAD tokens.

    Returns:
        list[str]: A list of decoded strings. (If Batch > 1, it flattens results or
                   can be modified to return list of lists).
    """
    decoded_texts = []

    # Iterate through each sentence step (N sentences)
    for seq_tensor in generated_sequences:
        # Iterate through the batch dimension (B)
        for i in range(len(seq_tensor)):
            # Convert tensor to python list of IDs
            token_ids = seq_tensor[i]
            # Decode
            text = tokenizer.decode(token_ids, skip_special_tokens=skip_special_tokens)
            decoded_texts.append(text)

    return decoded_texts


def decode_original_batch(input_ids, tokenizer):
    """
    Decodes a 3D tensor of input_ids into text.

    Args:
        input_ids (torch.Tensor): Shape (Batch_Size, Max_Sentences, Max_Words)
        tokenizer: HuggingFace tokenizer

    Returns:
        list[list[str]]: Nested list of strings [Batch][Sentence]
    """
    # Ensure it's on CPU and convert to list if it's a tensor
    if hasattr(input_ids, 'cpu'):
        input_ids = input_ids.cpu()

    batch_size, num_sentences, max_words = input_ids.shape
    all_decoded_text = []

    for b in range(batch_size):
        batch_sentences = []
        print(f"\n--- Batch Sample {b} ---")

        for n in range(num_sentences):
            # Get the sequence for this specific sentence
            sequence = input_ids[b, n]

            # Remove padding (optional, depending on tokenizer behavior)
            # sequence = sequence[sequence != tokenizer.pad_token_id]

            text = tokenizer.decode(sequence, skip_special_tokens=True)

            # Filter out empty strings if entire row was padding
            if text.strip():
                batch_sentences.append(text)

        all_decoded_text.append(batch_sentences)

    return all_decoded_text

In [ ]:
doc_eos = dec_tokenizer.eos_token_id                 # DOCUMENT end
bos_id = dec_tokenizer.bos_token_id
pad_token_id = dec_tokenizer.pad_token_id
sent_eos = dec_tokenizer.convert_tokens_to_ids("<EOS>")  # SENTENCE end
final_ids_tensor, all_generated_ids, all_logits, all_predicted_ids = generative_net.generate_autoregressive(
    test_zt,
    test_zi,
    real_ids,
    eos_sentence_id = sent_eos,
    eos_doc_id = doc_eos,
    bos_token_id = bos_id,
    pad_token_id = pad_token_id,
    max_words=50,
    prime_len = 0
)

In [ ]:
def calculate_nll_ignore_index(logits, real_ids, pad_token_id):
    # Pass ignore_index directly
    loss_fct = nn.CrossEntropyLoss(ignore_index=pad_token_id, reduction='mean')

    # Flatten inputs
    # logits: (Batch * Sent * Words, Vocab)
    # real_ids: (Batch * Sent * Words)
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = real_ids[..., 1:].contiguous()
    nll = loss_fct(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1)
    )

    return nll

In [ ]:
calculate_nll_ignore_index(all_logits, real_ids, pad_token_id)

In [ ]:
print(decode_original_batch(real_ids, dec_tokenizer))
print(decode_original_batch(all_predicted_ids, dec_tokenizer))

In [ ]:
# shuffling latents to see how the prediction comes out
test_zt1 = mu_t_q[1]
test_zi1 = mu_i_q[1]
test_zt1 = test_zt1.unsqueeze(0)
test_zi1 = test_zi1.unsqueeze(0)
real_ids1 = first_batch['dec_input_ids'][1].unsqueeze(0).to(device)

In [ ]:
test_zi1.shape

In [ ]:
predicted_ids1 = generative_net.inference_with_priming(test_zt1, test_zi1,real_ids,10, max_length=50, bos_token_id=dec_tokenizer.bos_token_id, eos_token_id=dec_tokenizer.eos_token_id)

In [ ]:
print(decode_original_batch(real_ids1, dec_tokenizer))
print(decode_generated_sequences(predicted_ids1,dec_tokenizer))

In [ ]:
# function to test ability of latents to discriminate between other sentences, un restricted
def test_latent_discrimination(encoder, decoder, dataloader, device, num_negatives=5):
    """
    Discrimination accuracy:
    For each (B,S) sentence slot that is real, count a win if
    NLL(x_true | z_true) < NLL(x_neg | z_true)
    """
    encoder.eval()
    decoder.eval()

    total = 0
    correct = 0

    print(f"Running Discrimination Test (1 Positive vs {num_negatives} Negatives)...")

    with torch.no_grad():
        for batch in dataloader:
            if isinstance(batch, dict):
                enc_input_ids = batch['enc_input_ids'].to(device)
                enc_word_mask = batch['enc_word_mask'].to(device)
                dec_input_ids = batch['dec_input_ids'].to(device)
                dec_word_mask = batch['dec_word_mask'].to(device)
            else:
                raise ValueError("Expected dict batch with enc/dec ids + masks")

            B, S, W = dec_input_ids.shape

            # Latents (use mu for a cleaner analysis if you want)
            mu_t, logvar_t, mu_i, logvar_i = encoder(dec_input_ids, dec_word_mask)
            z_t = reparameterize(mu_t, logvar_t)
            z_i = reparameterize(mu_i, logvar_i)

            # Positive
            pos_logits, _, _ = decoder(dec_input_ids, dec_word_mask, z_t, z_i)
            pos_nll, is_real = compute_sentence_nll(pos_logits, dec_input_ids, dec_word_mask)

            for _ in range(num_negatives):
                perm = torch.randperm(B, device=device)
                neg_ids  = dec_input_ids[perm]
                neg_mask = dec_word_mask[perm]

                neg_logits, _, _ = decoder(neg_ids, neg_mask, z_t, z_i)
                neg_nll, _ = compute_sentence_nll(neg_logits, neg_ids, neg_mask)

                # Compare only real sentences (from the POS sample slots)
                wins = (pos_nll < neg_nll) & is_real
                correct += wins.sum().item()
                total += is_real.sum().item()

    acc = correct / max(total, 1)
    print(f"Discrimination Accuracy: {acc:.2%}  (total comparisons: {total})")
    return acc

def compute_sentence_nll(logits, target_ids, mask, eps=1e-8):
    """
    Returns:
      sent_nll_avg: (B, S) average NLL per real token in the sentence
      sent_is_real: (B, S) bool mask: True if sentence has >=1 real token
    """
    B, S, W, V = logits.shape

    # Flatten
    flat_logits  = logits.reshape(-1, V)          # (B*S*W, V)
    flat_targets = target_ids.reshape(-1)         # (B*S*W,)
    flat_mask    = mask.reshape(-1).float()       # (B*S*W,)

    # Per-token CE
    per_token = F.cross_entropy(flat_logits, flat_targets, reduction='none')  # (B*S*W,)

    # Mask padding
    per_token = per_token * flat_mask

    # Back to (B, S, W)
    per_token = per_token.view(B, S, W)
    mask3     = mask.float().view(B, S, W)

    # Token counts per sentence
    tok_counts = mask3.sum(dim=2)                 # (B, S)
    sent_is_real = tok_counts > 0                 # (B, S)

    # Average NLL per token (avoid div0)
    sent_nll_sum = per_token.sum(dim=2)           # (B, S)
    sent_nll_avg = sent_nll_sum / (tok_counts + eps)

    return sent_nll_avg, sent_is_real

In [ ]:
test_latent_discrimination(inference_net,generative_net, train_loader,device, num_negatives=5)

In [ ]:
# function to see how the influence of latents on predictions for all batches in loader GOLD IDS IN HISTORY
def _per_example_kl(p_true, logp_true, logp_other):
    # p_true, logp_true, logp_other: (N, V)
    return (p_true * (logp_true - logp_other)).sum(dim=-1)

@torch.no_grad()
def plan_effect_curve_all_sentences(
    model,
    encoder,
    dataloader,
    device,
    max_steps=30,
    use_mu=True,
    shuffle_global=False,
    min_count=256,
    num_batches=None,   # None = all
):
    """
    Aggregates plan effect over a dataloader, using ALL real sentences in each batch.

    For each token step t (predict token t given tokens <t):
      ΔNLL(t) = NLL_shuf - NLL_true (positive => correct plan helps)
      acc_true(t), acc_shuf(t): next-token accuracy (argmax)
      KL_true_shuf(t) and KL_rand_rand(t) as controls

    Returns:
      dict with per-step mean curves + valid counts + suggested cutoff.
    """
    model.eval()
    encoder.eval()

    # Running sums (weighted by number of valid examples at that step)
    sum_delta = torch.zeros(max_steps, device=device)
    sum_nll_true = torch.zeros(max_steps, device=device)
    sum_nll_shuf = torch.zeros(max_steps, device=device)
    sum_kl_ts = torch.zeros(max_steps, device=device)
    sum_kl_rr = torch.zeros(max_steps, device=device)

    sum_acc_true = torch.zeros(max_steps, device=device)
    sum_acc_shuf = torch.zeros(max_steps, device=device)

    cnt = torch.zeros(max_steps, device=device)

    for b_idx, batch in enumerate(dataloader):
        if num_batches is not None and b_idx >= num_batches:
            break

        dec_input_ids = batch["dec_input_ids"].to(device)   # (B,S,W)
        dec_word_mask = batch["dec_word_mask"].to(device)   # (B,S,W)

        B, S, W = dec_input_ids.shape
        T = min(max_steps, W)

        # ----- Encode latents -----
        mu_t, logvar_t, mu_i, logvar_i = encoder(dec_input_ids, dec_word_mask)
        if use_mu:
            z_t, z_i = mu_t, mu_i
        else:
            z_t = mu_t + torch.randn_like(mu_t) * torch.exp(0.5 * logvar_t)
            z_i = mu_i + torch.randn_like(mu_i) * torch.exp(0.5 * logvar_i)

        # ----- Build plans (true) -----
        z_t_exp = z_t.unsqueeze(1).repeat(1, S, 1)                 # (B,S,Dz)
        gru_in_true = torch.cat([z_i, z_t_exp], dim=-1)            # (B,S,2Dz)
        h0 = model.gru_initial_projection(z_t).unsqueeze(0).repeat(model.gru.num_layers, 1, 1)
        plans_true, _ = model.gru(gru_in_true, h0)                 # (B,S,D_model)

        # ----- Build plans (shuf) -----
        perm = torch.randperm(B, device=device)
        z_i_shuf = z_i[perm]
        if shuffle_global:
            z_t_shuf = z_t[perm]
            z_t_exp_shuf = z_t_shuf.unsqueeze(1).repeat(1, S, 1)
        else:
            z_t_exp_shuf = z_t_exp
        plans_shuf, _ = model.gru(torch.cat([z_i_shuf, z_t_exp_shuf], dim=-1), h0)

        # ----- Random-vs-random control plans -----
        perm1 = torch.randperm(B, device=device)
        perm2 = torch.randperm(B, device=device)
        plans_r1, _ = model.gru(torch.cat([z_i[perm1], z_t_exp], dim=-1), h0)
        plans_r2, _ = model.gru(torch.cat([z_i[perm2], z_t_exp], dim=-1), h0)

        # ----- Flatten sentences: treat each sentence as an example -----
        ids = dec_input_ids.reshape(B * S, W)
        msk = dec_word_mask.reshape(B * S, W).bool()

        real_sent = msk.sum(dim=1) > 0
        if real_sent.sum().item() == 0:
            continue

        ids = ids[real_sent]
        msk = msk[real_sent]

        plans_true_f = plans_true.reshape(B * S, -1)[real_sent]
        plans_shuf_f = plans_shuf.reshape(B * S, -1)[real_sent]
        plans_r1_f   = plans_r1.reshape(B * S, -1)[real_sent]
        plans_r2_f   = plans_r2.reshape(B * S, -1)[real_sent]

        # Prefixes: (N, P, H)
        # helper to get logits
        def logits_from_prefix(curr_plan, hist_ids):
            # 1. Prepare Plan (Norm & Projections)
            norm_plan = model.plan_ln(curr_plan)

            # A. Calculate Embedding Bias (Gate * Plan)
            plan_emb = model.plan_to_emb(norm_plan)
            gate = model.plan_gate(norm_plan)
            # Shape: (B, 1, D) - Broadcastable across sequence length
            emb_bias = (gate * plan_emb).unsqueeze(1)

            # B. Calculate KV Prefix (Past)
            kv = model.plan_to_kv(norm_plan)
            bsz = curr_plan.shape[0]
            kv = kv.view(
                bsz,
                model.gpt2_n_layer,
                2,
                model.gpt2_n_head,
                model.prefix_len,
                model.gpt2_head_dim
            )
            kv = kv.permute(1, 2, 0, 3, 4, 5).contiguous()

            # Initialize Cache with Plan
            past = DynamicCache()
            for l in range(model.gpt2_n_layer):
                past.update(kv[l, 0], kv[l, 1], l)

            # 2. Prepare Inputs (Manual Embedding Injection)
            # Get raw token embeddings
            tok_emb = model.gpt2_model.wte(hist_ids)
            # Add the plan bias to every token in the history
            tok_emb = tok_emb + emb_bias

            # 3. Forward Pass
            # Use inputs_embeds + past_key_values
            gpt2_output = model.gpt2_model.transformer(
                inputs_embeds=tok_emb,
                past_key_values=past,
                use_cache=True
            )

            last_hidden_state = gpt2_outputout.last_hidden_state[:, -1, :]
            next_token_logits = model.final_linear(last_hidden_state)
            return next_token_logits

        # ----- Per-step eval -----
        for t in range(1, T):
            valid = msk[:, t]  # token t must be real
            n_valid = int(valid.sum().item())
            if n_valid == 0:
                continue

            hist_ids = ids[:, :t]                   # (N,t)
            tgt_ids  = ids[:, t]                    # (N,)


            logits_true = logits_from_prefix(plans_true_f, hist_ids)
            logits_shuf = logits_from_prefix(plans_shuf_f, hist_ids)
            logits_r1   = logits_from_prefix(plans_r1_f, hist_ids)
            logits_r2   = logits_from_prefix(plans_r2_f, hist_ids)

            # ---- NLL / ΔNLL (per-example then sum) ----
            nll_true_vec = F.cross_entropy(logits_true[valid], tgt_ids[valid], reduction="none")
            nll_shuf_vec = F.cross_entropy(logits_shuf[valid], tgt_ids[valid], reduction="none")
            delta_vec = nll_shuf_vec - nll_true_vec

            sum_nll_true[t] += nll_true_vec.sum()
            sum_nll_shuf[t] += nll_shuf_vec.sum()
            sum_delta[t] += delta_vec.sum()

            # ---- Accuracy ----
            pred_true = logits_true.argmax(dim=-1)
            pred_shuf = logits_shuf.argmax(dim=-1)
            acc_true_vec = (pred_true[valid] == tgt_ids[valid]).float()
            acc_shuf_vec = (pred_shuf[valid] == tgt_ids[valid]).float()

            sum_acc_true[t] += acc_true_vec.sum()
            sum_acc_shuf[t] += acc_shuf_vec.sum()

            # ---- KLs ----
            logp_true = F.log_softmax(logits_true[valid], dim=-1)
            p_true = logp_true.exp()
            logp_shuf = F.log_softmax(logits_shuf[valid], dim=-1)
            kl_ts_vec = _per_example_kl(p_true, logp_true, logp_shuf)
            sum_kl_ts[t] += kl_ts_vec.sum()

            logp_r1 = F.log_softmax(logits_r1[valid], dim=-1)
            p_r1 = logp_r1.exp()
            logp_r2 = F.log_softmax(logits_r2[valid], dim=-1)
            kl_rr_vec = _per_example_kl(p_r1, logp_r1, logp_r2)
            sum_kl_rr[t] += kl_rr_vec.sum()

            cnt[t] += n_valid

    eps = 1e-9
    counts = cnt.detach().cpu().long().tolist()

    mean_delta = (sum_delta / (cnt + eps)).detach().cpu().tolist()
    mean_nll_true = (sum_nll_true / (cnt + eps)).detach().cpu().tolist()
    mean_nll_shuf = (sum_nll_shuf / (cnt + eps)).detach().cpu().tolist()
    mean_kl_ts = (sum_kl_ts / (cnt + eps)).detach().cpu().tolist()
    mean_kl_rr = (sum_kl_rr / (cnt + eps)).detach().cpu().tolist()

    # Accuracy is already “sum correct / count”
    mean_acc_true = (sum_acc_true / (cnt + eps)).detach().cpu().tolist()
    mean_acc_shuf = (sum_acc_shuf / (cnt + eps)).detach().cpu().tolist()

    cutoff = 0
    for t in range(1, max_steps):
        if counts[t] >= min_count:
            cutoff = t

    return {
        "mean_delta_nll": mean_delta,
        "mean_nll_true": mean_nll_true,
        "mean_nll_shuf": mean_nll_shuf,
        "mean_acc_true": mean_acc_true,
        "mean_acc_shuf": mean_acc_shuf,
        "mean_kl_true_shuf": mean_kl_ts,
        "mean_kl_rand_rand": mean_kl_rr,
        "counts": counts,
        "cutoff_t": cutoff,
    }


In [ ]:
stats = plan_effect_curve_all_sentences(
    generative_net,
    inference_net,
    train_loader,
    device,
    max_steps=30,
    use_mu=True,
    shuffle_global=False,
    min_count=256,
    num_batches=None,   # None = all
)

In [ ]:
# function to plot the influence of latents on predictions
def plot_planning_metrics(stats):
    """
    Visualizes the output of plan_effect_curve_all_sentences.
    Expects 'stats' dictionary with keys: mean_delta_nll, mean_kl_true_shuf, etc.
    """

    # 1. Determine valid range based on cutoff
    cutoff = stats.get("cutoff_t", len(stats["mean_delta_nll"]) - 1)
    cutoff = min(cutoff, len(stats["mean_delta_nll"]) - 1)

    # X-axis: Token Steps (1 to cutoff)
    steps = np.arange(1, cutoff + 1)

    # Helper to slice data (ignoring index 0 which is usually empty/padding)
    def get_slice(key):
        return np.array(stats[key][1:cutoff+1])

    # 2. Setup Figure
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Latent Planning Horizon Diagnostics', fontsize=18, weight='bold')

    # --- PLOT A: Signal vs Noise (The "Loss of Control" Check) ---
    ax_kl = axes[0, 0]

    kl_signal = get_slice("mean_kl_true_shuf")
    kl_noise  = get_slice("mean_kl_rand_rand")

    ax_kl.plot(steps, kl_signal, label='Signal: KL(True || Shuffled)',
               color='#1f77b4', linewidth=2.5, marker='o', markersize=4)
    ax_kl.plot(steps, kl_noise, label='Noise: KL(Rand || Rand)',
               color='#7f7f7f', linestyle='--', linewidth=2)

    # Highlight failure point (Crossover)
    crossover = np.where(kl_signal < kl_noise)[0]
    if len(crossover) > 0:
        fail_step = steps[crossover[0]]
        ax_kl.axvline(x=fail_step, color='red', linestyle=':', alpha=0.8)
        ax_kl.text(fail_step + 0.5, ax_kl.get_ylim()[1]*0.9,
                   f'Signal Lost @ t={fail_step}', color='red', fontweight='bold')
        ax_kl.fill_between(steps, kl_signal, kl_noise,
                           where=(kl_noise > kl_signal),
                           color='red', alpha=0.1, label='Collapse Region')

    ax_kl.set_title("1. Latent Signal vs. Background Noise", fontsize=12)
    ax_kl.set_ylabel("KL Divergence (nats)")
    ax_kl.set_xlabel("Token Step")
    ax_kl.legend()
    ax_kl.grid(True, alpha=0.3)

    # --- PLOT B: Latent Utility (The "Exposure Bias" Check) ---
    ax_acc = axes[0, 1]

    acc_true = get_slice("mean_acc_true")
    acc_shuf = get_slice("mean_acc_shuf")

    ax_acc.plot(steps, acc_true, label='Acc w/ Correct Plan',
                color='#2ca02c', linewidth=2.5)
    ax_acc.plot(steps, acc_shuf, label='Acc w/ WRONG Plan',
                color='#ff7f0e', linestyle='--', linewidth=2)

    # Fill the "Benefit" gap
    ax_acc.fill_between(steps, acc_true, acc_shuf, color='#2ca02c', alpha=0.15,
                        label='Net Latent Benefit')

    ax_acc.set_title("2. Prediction Accuracy (Teacher Forcing)", fontsize=12)
    ax_acc.set_ylabel("Next-Token Accuracy")
    ax_acc.set_xlabel("Token Step")
    ax_acc.legend()
    ax_acc.grid(True, alpha=0.3)

    # --- PLOT C: Helpfulness (The "Gain" Check) ---
    ax_delta = axes[1, 0]

    delta_nll = get_slice("mean_delta_nll")

    # Color coding: Green (Helpful), Red (Harmful)
    colors = ['#2ca02c' if x > 0 else '#d62728' for x in delta_nll]

    ax_delta.bar(steps, delta_nll, color=colors, alpha=0.7, width=0.8)
    ax_delta.axhline(0, color='black', linewidth=1)

    ax_delta.set_title("3. Latent Helpfulness (ΔNLL)", fontsize=12)
    ax_delta.set_ylabel("NLL Improvement (nats)")
    ax_delta.set_xlabel("Token Step")
    ax_delta.grid(True, axis='y', alpha=0.3)

    # --- PLOT D: Reliability (Sample Count) ---
    ax_cnt = axes[1, 1]

    counts = get_slice("counts")

    ax_cnt.fill_between(steps, 0, counts, color='black', alpha=0.1)
    ax_cnt.plot(steps, counts, color='black', marker='.', linestyle='-')

    ax_cnt.set_title("4. Sample Validity (Count of Active Sentences)", fontsize=12)
    ax_cnt.set_ylabel("Number of Samples")
    ax_cnt.set_xlabel("Token Step")
    ax_cnt.grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# --- HOW TO USE ---
# plot_planning_metrics(stats)

In [ ]:
plot_planning_metrics(stats)

In [ ]:
stats

In [ ]:
# waht if feed first k tokens of gold can the model predict the rest
@torch.no_grad()
def test_first_k_tokens_gold_history(
    model, encoder, dataloader, device,
    K=8, bos_id=50256, max_batches=None,
    use_mu=True, shuffle_global=False, skip_bos=True,
):
    model.eval(); encoder.eval()

    sum_nll_true = 0.0
    sum_nll_shuf = 0.0
    sum_acc_true = 0.0
    sum_acc_shuf = 0.0
    count = 0

    for b_idx, batch in enumerate(dataloader):
        if max_batches is not None and b_idx >= max_batches:
            break

        dec_input_ids = batch["dec_input_ids"].to(device)
        dec_word_mask = batch["dec_word_mask"].to(device)

        out = _compute_plans_flat(model, encoder, dec_input_ids, dec_word_mask,
                                  use_mu=use_mu, shuffle_global=shuffle_global)
        if out is None:
            continue
        ids, msk, plan_true, plan_shuf = out  # (N,W), (N,W), (N,D)
        N, W = ids.shape

        first_idx, has_tok = _first_content_index(ids, msk, bos_id=bos_id, skip_bos=skip_bos)
        if has_tok.sum().item() == 0:
            continue

        ids = ids[has_tok]
        msk = msk[has_tok]
        first_idx = first_idx[has_tok]
        plan_true = plan_true[has_tok]
        plan_shuf = plan_shuf[has_tok]



        # loop over next K tokens after first_idx
        for j in range(K):
            t = first_idx + j
            valid = t < W
            if valid.sum().item() == 0:
                break

            t_idx = t[valid]
            ids_v = ids[valid]
            msk_v = msk[valid]
            plan_true_v = plan_true[valid]
            plan_shuf_v = plan_shuf[valid]

            # token at position t must be real
            ok = msk_v[torch.arange(ids_v.size(0), device=device), t_idx]
            if ok.sum().item() == 0:
                continue

            ids_ok = ids_v[ok]
            t_ok = t_idx[ok]
            plan_true_ok = plan_true_v[ok]
            plan_shuf_ok = plan_shuf_v[ok]

            # history is gold tokens up to t (exclusive)
            # per-example variable t, so we do a small loop (K is small)
            for n in range(ids_ok.size(0)):
                tt = int(t_ok[n].item())
                if tt == 0:
                    # no history: prefix-only
                    logits_true = _logits_next_from_prefix_only(model, plan_true_ok[n:n+1],bos_id)
                    logits_shuf = _logits_next_from_prefix_only(model, plan_shuf_ok[n:n+1],bos_id)
                else:
                    logits_true = _logits_next_from_prefix_and_history(model, plan_true_ok[n:n+1], ids_ok[n:n+1, :tt])
                    logits_shuf = _logits_next_from_prefix_and_history(model, plan_shuf_ok[n:n+1], ids_ok[n:n+1, :tt])

                target = ids_ok[n, tt].view(1)

                nll_true = F.cross_entropy(logits_true, target, reduction="sum").item()
                nll_shuf = F.cross_entropy(logits_shuf, target, reduction="sum").item()

                sum_nll_true += nll_true
                sum_nll_shuf += nll_shuf

                sum_acc_true += float((logits_true.argmax(-1) == target).item())
                sum_acc_shuf += float((logits_shuf.argmax(-1) == target).item())
                count += 1

    eps = 1e-9
    return {
        "K": K,
        "count": count,
        "mean_nll_true": sum_nll_true / (count + eps),
        "mean_nll_shuf": sum_nll_shuf / (count + eps),
        "mean_delta_nll": (sum_nll_shuf - sum_nll_true) / (count + eps),
        "mean_acc_true": sum_acc_true / (count + eps),
        "mean_acc_shuf": sum_acc_shuf / (count + eps),
    }

In [ ]:
# First token anchoring and Noisy history plan
# -------------------------
# helpers
# -------------------------
def _first_content_index(ids, msk, bos_id=50256, skip_bos=True):
    """
    ids, msk: (N,W), msk bool
    Returns:
      first_idx: (N,) index of first "content" token
      has_token: (N,) bool
    """
    N, W = ids.shape
    pos = torch.arange(W, device=ids.device).unsqueeze(0).expand(N, W)

    valid = msk
    if skip_bos:
        valid = valid & (ids != bos_id)

    big = torch.full_like(pos, W)
    first_idx = torch.where(valid, pos, big).min(dim=1).values
    has_token = first_idx < W
    return first_idx, has_token


def _compute_plans_flat(model, encoder, dec_input_ids, dec_word_mask, use_mu=True, shuffle_global=False):
    """
    Returns flattened per-sentence:
      ids, msk: (N,W)
      plan_true, plan_shuf: (N,D_model)
    where N = (# real sentences in batch)
    """
    device = dec_input_ids.device
    B, S, W = dec_input_ids.shape

    mu_t, logvar_t, mu_i, logvar_i = encoder(dec_input_ids, dec_word_mask)
    if use_mu:
        z_t, z_i = mu_t, mu_i
    else:
        z_t = mu_t + torch.randn_like(mu_t) * torch.exp(0.5 * logvar_t)
        z_i = mu_i + torch.randn_like(mu_i) * torch.exp(0.5 * logvar_i)

    z_t_exp = z_t.unsqueeze(1).repeat(1, S, 1)
    h0 = model.gru_initial_projection(z_t).unsqueeze(0).repeat(model.gru.num_layers, 1, 1)

    # true plans
    plans_true, _ = model.gru(torch.cat([z_i, z_t_exp], dim=-1), h0)  # (B,S,Dm)

    # shuffled plans
    perm = torch.randperm(B, device=device)
    z_i_shuf = z_i[perm]
    if shuffle_global:
        z_t_shuf = z_t[perm]
        z_t_exp_shuf = z_t_shuf.unsqueeze(1).repeat(1, S, 1)
    else:
        z_t_exp_shuf = z_t_exp
    plans_shuf, _ = model.gru(torch.cat([z_i_shuf, z_t_exp_shuf], dim=-1), h0)

    # flatten sentences
    ids = dec_input_ids.reshape(B * S, W)
    msk = dec_word_mask.reshape(B * S, W).bool()
    real_sent = msk.sum(dim=1) > 0
    if real_sent.sum().item() == 0:
        return None

    ids = ids[real_sent]
    msk = msk[real_sent]
    plan_true = plans_true.reshape(B * S, -1)[real_sent]
    plan_shuf = plans_shuf.reshape(B * S, -1)[real_sent]
    return ids, msk, plan_true, plan_shuf


def _logits_next_from_prefix_and_history(model, curr_plan, history_ids):
    """
    curr_plan: (N, d_model) - Raw latent plan
    history_ids: (N, t) - Sequence of history token IDs
    returns logits: (N, V) for next token
    """
    bsz = curr_plan.size(0)

    # 1. Prepare Plan: Norm & Projections
    norm_plan = model.plan_ln(curr_plan)

    # A. Calculate Embedding Bias (Gate * Plan)
    plan_emb = model.plan_to_emb(norm_plan)
    gate = model.plan_gate(norm_plan)
    emb_bias = (gate * plan_emb).unsqueeze(1) # (N, 1, d_model)

    # B. Calculate KV Prefix (Past)
    kv = model.plan_to_kv(norm_plan)
    kv = kv.view(
        bsz, model.gpt2_n_layer, 2, model.gpt2_n_head,
        model.prefix_len, model.gpt2_head_dim
    )
    kv = kv.permute(1, 2, 0, 3, 4, 5).contiguous()

    # Initialize Cache
    past = DynamicCache()
    for l in range(model.gpt2_n_layer):
        past.update(kv[l, 0], kv[l, 1], l)

    # 2. Prepare Inputs (Manual Embedding Injection)
    tok_emb = model.gpt2_model.wte(history_ids)
    tok_emb = tok_emb + emb_bias # Inject plan into history embeddings

    # 3. Forward Pass
    gpt2_output = model.gpt2_model.transformer(
        inputs_embeds=tok_emb,
        past_key_values=past,
        use_cache=True
    )

    last_hidden_state = gpt2_output.last_hidden_state[:, -1, :]
    return model.final_linear(last_hidden_state)


def _logits_next_from_prefix_only(model, curr_plan, bos_id):
    """
    curr_plan: (N, d_model)
    bos_id: int
    returns logits: (N, V) for next token after prefix (and BOS)
    """
    bsz = curr_plan.size(0)
    device = curr_plan.device

    # 1. Prepare Plan (Same as above)
    norm_plan = model.plan_ln(curr_plan)

    plan_emb = model.plan_to_emb(norm_plan)
    gate = model.plan_gate(norm_plan)
    emb_bias = (gate * plan_emb).unsqueeze(1)

    kv = model.plan_to_kv(norm_plan)
    kv = kv.view(
        bsz, model.gpt2_n_layer, 2, model.gpt2_n_head,
        model.prefix_len, model.gpt2_head_dim
    )
    kv = kv.permute(1, 2, 0, 3, 4, 5).contiguous()

    past = DynamicCache()
    for l in range(model.gpt2_n_layer):
        past.update(kv[l, 0], kv[l, 1], l)

    # 2. Prepare Inputs (BOS Token + Injection)
    input_ids = torch.full((bsz, 1), bos_id, dtype=torch.long, device=device)
    tok_emb = model.gpt2_model.wte(input_ids)
    tok_emb = tok_emb + emb_bias

    # 3. Forward Pass
    gpt2_output = model.gpt2_model.transformer(
        inputs_embeds=tok_emb,
        past_key_values=past,
        use_cache=True
    )

    last_hidden_state = gpt2_output.last_hidden_state[:, -1, :]
    return model.final_linear(last_hidden_state)

# ============================================================
# 1) First-token anchoring test (NO GOLD HISTORY; prefix only)
# ============================================================
@torch.no_grad()
def test_first_token_anchoring(
    model,
    encoder,
    dataloader,
    device,
    bos_id=50256,
    max_batches=None,
    use_mu=True,
    shuffle_global=False,
    skip_bos=True,
):
    """
    Predict the FIRST content token in each real sentence using ONLY the plan-prefix (no history).

    Returns:
      mean_nll_true, mean_nll_shuf, mean_delta_nll,
      mean_acc_true, mean_acc_shuf,
      count
    """
    model.eval(); encoder.eval()

    sum_nll_true = 0.0
    sum_nll_shuf = 0.0
    sum_acc_true = 0.0
    sum_acc_shuf = 0.0
    count = 0

    for b_idx, batch in enumerate(dataloader):
        if max_batches is not None and b_idx >= max_batches:
            break

        dec_input_ids = batch["dec_input_ids"].to(device)
        dec_word_mask = batch["dec_word_mask"].to(device)

        out = _compute_plans_flat(model, encoder, dec_input_ids, dec_word_mask,
                                  use_mu=use_mu, shuffle_global=shuffle_global)
        if out is None:
            continue
        ids, msk, plan_true, plan_shuf = out
        N, W = ids.shape

        # find first content token per sentence
        first_idx, has_tok = _first_content_index(ids, msk, bos_id=bos_id, skip_bos=skip_bos)
        if has_tok.sum().item() == 0:
            continue

        ids = ids[has_tok]
        first_idx = first_idx[has_tok]
        plan_true = plan_true[has_tok]
        plan_shuf = plan_shuf[has_tok]

        # targets: first content tokens
        targets = ids[torch.arange(ids.size(0), device=device), first_idx]  # (N,)


        logits_true = _logits_next_from_prefix_only(model, plan_true,bos_id)
        logits_shuf = _logits_next_from_prefix_only(model, plan_shuf,bos_id)

        nll_true = F.cross_entropy(logits_true, targets, reduction="none")
        nll_shuf = F.cross_entropy(logits_shuf, targets, reduction="none")

        sum_nll_true += float(nll_true.sum().item())
        sum_nll_shuf += float(nll_shuf.sum().item())

        pred_true = logits_true.argmax(dim=-1)
        pred_shuf = logits_shuf.argmax(dim=-1)
        sum_acc_true += float((pred_true == targets).float().sum().item())
        sum_acc_shuf += float((pred_shuf == targets).float().sum().item())

        count += int(targets.numel())

    eps = 1e-9
    mean_nll_true = sum_nll_true / (count + eps)
    mean_nll_shuf = sum_nll_shuf / (count + eps)

    return {
        "count": count,
        "mean_nll_true": mean_nll_true,
        "mean_nll_shuf": mean_nll_shuf,
        "mean_delta_nll": mean_nll_shuf - mean_nll_true,
        "mean_acc_true": sum_acc_true / (count + eps),
        "mean_acc_shuf": sum_acc_shuf / (count + eps),
    }


# ============================================================
# 2) Noisy-history curve (corrupt history, then evaluate)
# ============================================================
@torch.no_grad()
def noisy_history_plan_effect_curve(
    model,
    encoder,
    dataloader,
    device,
    max_steps=30,
    p_corrupt=0.2,
    corruption_mode="model_greedy",  # "model_greedy" | "random"
    target_mode="gold",              # "gold" | "noisy"
    bos_id=50256,
    use_mu=True,
    shuffle_global=False,
    min_count=256,
    max_batches=None,
):
    """
    Build a noisy version of each sentence by corrupting tokens as you roll forward.

    corruption_mode:
      - "model_greedy": when corrupting position j, replace with argmax token under TRUE plan given noisy history <j
      - "random": replace with uniform random token id (0..V-1)

    target_mode:
      - "gold": evaluate NLL/acc on the gold next token (can become inconsistent with noisy history)
      - "noisy": evaluate NLL/acc on the noisy next token (tests self-consistency on perturbed trajectory)

    Returns per-step curves (only trust up to cutoff_t where counts>=min_count).
    """
    model.eval(); encoder.eval()

    sum_delta = torch.zeros(max_steps, device=device)
    sum_nll_true = torch.zeros(max_steps, device=device)
    sum_nll_shuf = torch.zeros(max_steps, device=device)
    sum_acc_true = torch.zeros(max_steps, device=device)
    sum_acc_shuf = torch.zeros(max_steps, device=device)
    cnt = torch.zeros(max_steps, device=device)

    # vocab size for random corruption
    # (assumes final_linear outputs vocab logits)
    V = model.final_linear.out_features

    for b_idx, batch in enumerate(dataloader):
        if max_batches is not None and b_idx >= max_batches:
            break

        dec_input_ids = batch["dec_input_ids"].to(device)
        dec_word_mask = batch["dec_word_mask"].to(device)

        out = _compute_plans_flat(model, encoder, dec_input_ids, dec_word_mask,
                                  use_mu=use_mu, shuffle_global=shuffle_global)
        if out is None:
            continue
        ids, msk, plan_true, plan_shuf = out  # (N,W), (N,W), (N,Dm), (N,Dm)
        N, W = ids.shape
        T = min(max_steps, W)



        # ---- build noisy sequence (right-pad aware) ----
        noisy = ids.clone()

        # keep bos as-is if present; otherwise start at j=0
        start_j = 1 if (W > 0 and (noisy[:, 0] == bos_id).all().item()) else 0

        for j in range(start_j, T):
            valid_j = msk[:, j]  # token position j exists?
            if valid_j.sum().item() == 0:
                continue

            corrupt_mask = (torch.rand(N, device=device) < p_corrupt) & valid_j
            if corrupt_mask.sum().item() == 0:
                continue

            if corruption_mode == "random":
                noisy[corrupt_mask, j] = torch.randint(0, V, (int(corrupt_mask.sum().item()),), device=device)
            elif corruption_mode == "model_greedy":
                # predict token at position j given noisy history <j under TRUE plan
                if j == 0:
                    # no history; use prefix-only
                    logits_next = _logits_next_from_prefix_only(model, plan_true, bos_id)
                else:
                    logits_next = _logits_next_from_prefix_and_history(model, plan_true, noisy[:, :j])
                pred = logits_next.argmax(dim=-1)
                noisy[corrupt_mask, j] = pred[corrupt_mask]
            else:
                raise ValueError(f"Unknown corruption_mode: {corruption_mode}")

        # ---- evaluate per-step next-token prediction under noisy history ----
        for t in range(1, T):
            valid_t = msk[:, t]
            n_valid = int(valid_t.sum().item())
            if n_valid == 0:
                continue

            hist = noisy[:, :t]  # (N,t)

            logits_true = _logits_next_from_prefix_and_history(model, plan_true, hist)
            logits_shuf = _logits_next_from_prefix_and_history(model, plan_shuf, hist)

            targets = ids[:, t] if target_mode == "gold" else noisy[:, t]

            nll_true_vec = F.cross_entropy(logits_true[valid_t], targets[valid_t], reduction="none")
            nll_shuf_vec = F.cross_entropy(logits_shuf[valid_t], targets[valid_t], reduction="none")

            sum_nll_true[t] += nll_true_vec.sum()
            sum_nll_shuf[t] += nll_shuf_vec.sum()
            sum_delta[t] += (nll_shuf_vec - nll_true_vec).sum()

            pred_true = logits_true.argmax(dim=-1)
            pred_shuf = logits_shuf.argmax(dim=-1)
            sum_acc_true[t] += (pred_true[valid_t] == targets[valid_t]).float().sum()
            sum_acc_shuf[t] += (pred_shuf[valid_t] == targets[valid_t]).float().sum()

            cnt[t] += n_valid

    eps = 1e-9
    counts = cnt.detach().cpu().long().tolist()

    mean_delta = (sum_delta / (cnt + eps)).detach().cpu().tolist()
    mean_nll_true = (sum_nll_true / (cnt + eps)).detach().cpu().tolist()
    mean_nll_shuf = (sum_nll_shuf / (cnt + eps)).detach().cpu().tolist()
    mean_acc_true = (sum_acc_true / (cnt + eps)).detach().cpu().tolist()
    mean_acc_shuf = (sum_acc_shuf / (cnt + eps)).detach().cpu().tolist()

    cutoff = 0
    for t in range(1, max_steps):
        if counts[t] >= min_count:
            cutoff = t

    return {
        "p_corrupt": p_corrupt,
        "corruption_mode": corruption_mode,
        "target_mode": target_mode,
        "mean_delta_nll": mean_delta,
        "mean_nll_true": mean_nll_true,
        "mean_nll_shuf": mean_nll_shuf,
        "mean_acc_true": mean_acc_true,
        "mean_acc_shuf": mean_acc_shuf,
        "counts": counts,
        "cutoff_t": cutoff,
    }


In [ ]:
def plot_planning_metrics_safe(stats):
    """
    Adapted to handle missing KL keys and 'count' vs 'counts' mismatch.
    """

    # 1. Handle Key Mismatches
    # Map 'count' to 'counts' if necessary
    if "counts" not in stats and "count" in stats:
        stats["counts"] = stats["count"]

    # 2. Determine valid range
    # Assumes stats are LISTS. If scalars, this len() check will fail or be 1.
    cutoff = len(stats["mean_delta_nll"])

    # X-axis: Token Steps
    steps = np.arange(1, cutoff + 1)

    # Helper to slice data (adjust slice depending on if you have padding at index 0)
    # If your lists start directly at t=1, use [0:cutoff]. If index 0 is padding, use [1:cutoff+1].
    # This version assumes input lists match 'steps' length exactly.
    def get_slice(key):
        data = stats.get(key, np.zeros(cutoff)) # Fallback to zeros if missing
        return np.array(data)

    # 3. Setup Figure
    # We hide Plot A (KL) if keys are missing
    has_kl = "mean_kl_true_shuf" in stats

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Latent Planning Horizon Diagnostics', fontsize=18, weight='bold')

    # --- PLOT A: Signal vs Noise (KL) ---
    ax_kl = axes[0, 0]
    if has_kl:
        kl_signal = get_slice("mean_kl_true_shuf")
        kl_noise  = get_slice("mean_kl_rand_rand")
        ax_kl.plot(steps, kl_signal, label='Signal', color='#1f77b4', marker='o')
        ax_kl.plot(steps, kl_noise, label='Noise', color='#7f7f7f', linestyle='--')
        ax_kl.legend()
    else:
        ax_kl.text(0.5, 0.5, "KL Data Missing", ha='center', fontsize=14, color='gray')
    ax_kl.set_title("1. Latent Signal (KL)")
    ax_kl.set_xlabel("Token Step")
    ax_kl.grid(True, alpha=0.3)

    # --- PLOT B: Latent Utility (Accuracy) ---
    ax_acc = axes[0, 1]
    acc_true = get_slice("mean_acc_true")
    acc_shuf = get_slice("mean_acc_shuf")

    ax_acc.plot(steps, acc_true, label='Acc w/ Plan', color='#2ca02c')
    ax_acc.plot(steps, acc_shuf, label='Acc w/o Plan', color='#ff7f0e', linestyle='--')
    ax_acc.fill_between(steps, acc_true, acc_shuf, color='#2ca02c', alpha=0.15)
    ax_acc.set_title("2. Prediction Accuracy")
    ax_acc.legend()
    ax_acc.grid(True, alpha=0.3)

    # --- PLOT C: Helpfulness (Delta NLL) ---
    ax_delta = axes[1, 0]
    delta_nll = get_slice("mean_delta_nll")
    colors = ['#2ca02c' if x > 0 else '#d62728' for x in delta_nll]
    ax_delta.bar(steps, delta_nll, color=colors, alpha=0.7)
    ax_delta.axhline(0, color='black', linewidth=1)
    ax_delta.set_title("3. Latent Helpfulness (Delta NLL)")
    ax_delta.grid(True, axis='y', alpha=0.3)

    # --- PLOT D: Reliability (Counts) ---
    ax_cnt = axes[1, 1]
    counts = get_slice("counts")
    ax_cnt.plot(steps, counts, color='black', marker='.')
    ax_cnt.set_title("4. Sample Validity (Count)")
    ax_cnt.grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [ ]:
stats1 = test_first_k_tokens_gold_history(
    generative_net, inference_net, train_loader, device,
    K=8, bos_id=dec_tokenizer.bos_token_id, max_batches=None,
    use_mu=True, shuffle_global=False, skip_bos=True,
)

In [ ]:
stats1

In [ ]:
stats2 = noisy_history_plan_effect_curve(
    generative_net,
    inference_net,
    train_loader,
    device,
    max_steps=30,
    p_corrupt=0.2,
    corruption_mode="model_greedy",  # "model_greedy" | "random"
    target_mode="gold",              # "gold" | "noisy"
    bos_id=dec_tokenizer.bos_token_id,
    use_mu=True,
    shuffle_global=False,
    min_count=256,
    max_batches=None,
)

In [ ]:
plot_planning_metrics_safe(stats2)

In [ ]:
stats2

In [ ]:
# Latents only roll out
@torch.no_grad()
def prefix_only_rollout_anchoring_curve(
    model,
    encoder,
    dataloader,
    device,
    K=20,
    bos_id=50256,
    use_mu=True,
    shuffle_global=False,
    skip_bos=True,
    max_batches=None,
    greedy=True,          # if False, sample
    temperature=1.0,
    top_k=None,
    min_count=256,
):
    """
    HARD test: prefix-only rollout.
    For each sentence:
      - start from prefix only (no gold history)
      - generate tokens step by step (ŷ_0, ŷ_1, ...)
      - at each step j, evaluate how well it predicts the GOLD token x_j
        *given its own generated history*.

    Computes per-step (j=0..K-1):
      - NLL_true / NLL_shuf on gold token
      - ΔNLL = NLL_shuf - NLL_true
      - acc_next_true / acc_next_shuf: argmax matches gold token (next-token under that history)
      - gen_match_true / gen_match_shuf: generated token equals gold token (actual rollout match)
      - counts[j]

    Notes:
      - Uses first content token index per sentence (skips BOS if skip_bos=True).
      - Ignores padding via masks and stops contributing when gold token doesn't exist.
    """
    model.eval(); encoder.eval()

    # running sums per step
    sum_nll_true = torch.zeros(K, device=device)
    sum_nll_shuf = torch.zeros(K, device=device)
    sum_delta    = torch.zeros(K, device=device)

    sum_acc_next_true = torch.zeros(K, device=device)
    sum_acc_next_shuf = torch.zeros(K, device=device)

    sum_gen_match_true = torch.zeros(K, device=device)
    sum_gen_match_shuf = torch.zeros(K, device=device)

    cnt = torch.zeros(K, device=device)

    # vocab size
    V = model.final_linear.out_features

    def sample_from_logits(logits):
        # logits: (N,V)
        if temperature != 1.0:
            logits = logits / max(temperature, 1e-6)

        if top_k is not None and top_k > 0:
            topk_vals, topk_idx = torch.topk(logits, k=min(top_k, logits.size(-1)), dim=-1)
            probs = F.softmax(topk_vals, dim=-1)
            choice = torch.multinomial(probs, num_samples=1).squeeze(-1)
            return topk_idx[torch.arange(logits.size(0), device=logits.device), choice]
        else:
            probs = F.softmax(logits, dim=-1)
            return torch.multinomial(probs, num_samples=1).squeeze(-1)

    for b_idx, batch in enumerate(dataloader):
        if max_batches is not None and b_idx >= max_batches:
            break

        dec_input_ids = batch["dec_input_ids"].to(device)   # (B,S,W)
        dec_word_mask = batch["dec_word_mask"].to(device)   # (B,S,W)

        out = _compute_plans_flat(
            model, encoder, dec_input_ids, dec_word_mask,
            use_mu=use_mu, shuffle_global=shuffle_global
        )
        if out is None:
            continue

        ids, msk, plan_true, plan_shuf = out   # ids/msk: (N,W)
        N, W = ids.shape

        # First content token index (per sentence)
        first_idx, has_tok = _first_content_index(ids, msk, bos_id=bos_id, skip_bos=skip_bos)
        if has_tok.sum().item() == 0:
            continue

        ids = ids[has_tok]
        msk = msk[has_tok]
        first_idx = first_idx[has_tok]
        plan_true = plan_true[has_tok]
        plan_shuf = plan_shuf[has_tok]

        N = ids.size(0)

        # Prefix embeddings


        # We'll store generated histories here (N,K)
        # Values for inactive rows don't matter because we'll mask them out via "active".
        gen_hist_true = torch.full((N, K), bos_id, device=device, dtype=ids.dtype)
        gen_hist_shuf = torch.full((N, K), bos_id, device=device, dtype=ids.dtype)

        for j in range(K):
            gold_pos = first_idx + j  # (N,)

            # which examples still have a real gold token at this position?
            in_bounds = gold_pos < W
            if in_bounds.sum().item() == 0:
                break

            idx_in = torch.nonzero(in_bounds, as_tuple=False).squeeze(-1)
            gold_pos_in = gold_pos[idx_in]
            msk_in = msk[idx_in]

            # real token mask at that gold position
            real = msk_in[torch.arange(idx_in.numel(), device=device), gold_pos_in]
            if real.sum().item() == 0:
                continue

            idx = idx_in[real]
            gold_pos_ok = gold_pos_in[real]

            # gold target token
            gold_tok = ids[idx, gold_pos_ok]  # (M,)

            # Build history = generated tokens so far (length j)
            if j == 0:
                logits_true = _logits_next_from_prefix_only(model, plan_true[idx],bos_id)
                logits_shuf = _logits_next_from_prefix_only(model, plan_shuf[idx],bos_id)
            else:
                hist_true = gen_hist_true[idx, :j]
                hist_shuf = gen_hist_shuf[idx, :j]
                logits_true = _logits_next_from_prefix_and_history(model, plan_true[idx], hist_true)
                logits_shuf = _logits_next_from_prefix_and_history(model, plan_shuf[idx], hist_shuf)

            # NLL on GOLD token under each condition
            nll_t = F.cross_entropy(logits_true, gold_tok, reduction="none")
            nll_s = F.cross_entropy(logits_shuf, gold_tok, reduction="none")

            sum_nll_true[j] += nll_t.sum()
            sum_nll_shuf[j] += nll_s.sum()
            sum_delta[j]    += (nll_s - nll_t).sum()

            # next-token accuracy (argmax == gold token)
            pred_next_true = logits_true.argmax(dim=-1)
            pred_next_shuf = logits_shuf.argmax(dim=-1)
            sum_acc_next_true[j] += (pred_next_true == gold_tok).float().sum()
            sum_acc_next_shuf[j] += (pred_next_shuf == gold_tok).float().sum()

            # choose generated token for rollout (greedy or sample)
            if greedy:
                gen_tok_true = pred_next_true
                gen_tok_shuf = pred_next_shuf
            else:
                gen_tok_true = sample_from_logits(logits_true)
                gen_tok_shuf = sample_from_logits(logits_shuf)

            # rollout-match vs gold (actual generated token equals gold)
            sum_gen_match_true[j] += (gen_tok_true == gold_tok).float().sum()
            sum_gen_match_shuf[j] += (gen_tok_shuf == gold_tok).float().sum()

            # write generated token into history buffer
            gen_hist_true[idx, j] = gen_tok_true
            gen_hist_shuf[idx, j] = gen_tok_shuf

            cnt[j] += gold_tok.numel()

    eps = 1e-9
    counts = cnt.detach().cpu().long().tolist()

    mean_delta = (sum_delta / (cnt + eps)).detach().cpu().tolist()
    mean_nll_t = (sum_nll_true / (cnt + eps)).detach().cpu().tolist()
    mean_nll_s = (sum_nll_shuf / (cnt + eps)).detach().cpu().tolist()

    mean_acc_next_t = (sum_acc_next_true / (cnt + eps)).detach().cpu().tolist()
    mean_acc_next_s = (sum_acc_next_shuf / (cnt + eps)).detach().cpu().tolist()

    mean_gen_match_t = (sum_gen_match_true / (cnt + eps)).detach().cpu().tolist()
    mean_gen_match_s = (sum_gen_match_shuf / (cnt + eps)).detach().cpu().tolist()

    cutoff = 0
    for j in range(K):
        if counts[j] >= min_count:
            cutoff = j

    return {
        "K": K,
        "greedy": greedy,
        "counts": counts,
        "cutoff_j": cutoff,
        "mean_delta_nll": mean_delta,
        "mean_nll_true": mean_nll_t,
        "mean_nll_shuf": mean_nll_s,
        "mean_acc_next_true": mean_acc_next_t,
        "mean_acc_next_shuf": mean_acc_next_s,
        "mean_gen_match_true": mean_gen_match_t,
        "mean_gen_match_shuf": mean_gen_match_s,
    }



In [ ]:
stats3 = prefix_only_rollout_anchoring_curve(
    generative_net,
    inference_net,
    train_loader,
    device,
    K=20,
    bos_id=dec_tokenizer.bos_token_id,
    use_mu=True,
    shuffle_global=False,
    skip_bos=True,
    max_batches=None,
    greedy=True,          # if False, sample
    temperature=1.0,
    top_k=None,
    min_count=256,
)

In [ ]:
plot_planning_metrics_safe(stats3)

In [ ]:
stats3

In [ ]:
# Printing Latents neighbors
def reparameterize(mu, sigma2, eps=1e-8):
    sigma2 = torch.clamp(sigma2, min=eps)
    return mu + torch.randn_like(mu) * torch.sqrt(sigma2)

def test_latent_neighbors(encoder, dataloader, tokenizer, device,
                          num_examples=5, use_mu_for_analysis=True):
    """
    Finds nearest neighbors in local latent space (z_i) for REAL (non-padding) sentences.
    Uses cosine distance.
    """
    encoder.eval()

    all_latents = []
    all_texts = []

    with torch.no_grad():
        for batch in dataloader:
            if not isinstance(batch, dict):
                raise ValueError("Expected dict batch with enc/dec ids + masks.")

            enc_input_ids = batch['enc_input_ids'].to(device)
            enc_word_mask = batch['enc_word_mask'].to(device)

            # Prefer decoder-side ids/mask for filtering/decoding if you have them
            dec_input_ids = batch.get('dec_input_ids', enc_input_ids).to(device)
            dec_word_mask = batch.get('dec_word_mask', enc_word_mask).to(device)

            mu_t, sigma2_t, mu_i, sigma2_i = encoder(dec_input_ids, dec_word_mask)

            # For analysis, mu is often cleaner than samples
            z_i = mu_i if use_mu_for_analysis else reparameterize(mu_i, sigma2_i)

            B, S, D = z_i.shape
            _, _, W = dec_input_ids.shape

            flat_latents = z_i.reshape(B * S, D)          # (B*S, D)
            flat_ids     = dec_input_ids.reshape(B * S, W)

            # Real sentence mask from decoder mask (more consistent with decoded ids)
            flat_m = dec_word_mask.reshape(B * S, W)
            is_real = flat_m.sum(dim=1) > 0               # (B*S,)

            # Keep only real sentences
            flat_latents = flat_latents[is_real]
            flat_ids     = flat_ids[is_real]

            # Now build aligned (latent, text) pairs
            for latent_vec, seq in zip(flat_latents, flat_ids):
                text = tokenizer.decode(seq.tolist(), skip_special_tokens=True).strip()
                if not text:
                    continue
                all_latents.append(latent_vec.detach().cpu().numpy())
                all_texts.append(text)

    if len(all_latents) < 10:
        print(f"Too few valid samples: {len(all_latents)}")
        return None

    all_latents = np.stack(all_latents, axis=0)

    all_latents = all_latents / (np.linalg.norm(all_latents, axis=1, keepdims=True) + 1e-8)


    print(f"Fitting NN on {len(all_latents)} valid samples...")
    nbrs = NearestNeighbors(n_neighbors=4, metric='cosine').fit(all_latents)

    num_examples = min(num_examples, len(all_latents))
    indices = np.random.choice(len(all_latents), num_examples, replace=False)

    for idx in indices:
        query_text = all_texts[idx]
        query_latent = all_latents[idx].reshape(1, -1)

        distances, neighbor_indices = nbrs.kneighbors(query_latent)

        print(f"\nQuery: {query_text}")
        print("-" * 50)

        found = 0
        for dist, nidx in zip(distances[0], neighbor_indices[0]):
            if nidx == idx:
                continue
            print(f"Neighbor {found+1} (Dist: {dist:.4f}): {all_texts[nidx]}")
            found += 1
            if found == 3:
                break

    return nbrs

In [ ]:
test_latent_neighbors(inference_net, train_loader, dec_tokenizer, device,num_examples=5, use_mu_for_analysis=False)

In [ ]:
# Sentence nll avergaed per token
def compute_sentence_nll_avg(logits, target_ids, mask, eps=1e-8):
    """
    logits: (B,S,W,V)
    target_ids: (B,S,W)
    mask: (B,S,W) 1 for real tokens, 0 for pad
    Returns:
      nll_avg: (B,S)  average NLL per real token
      is_real: (B,S)  True if sentence has >=1 real token
    """
    B, S, W, V = logits.shape
    flat_logits = logits.reshape(-1, V)
    flat_tgt = target_ids.reshape(-1)
    flat_mask = mask.reshape(-1).float()

    per_tok = F.cross_entropy(flat_logits, flat_tgt, reduction="none") * flat_mask
    per_tok = per_tok.view(B, S, W)
    mask3 = mask.float()

    tok_counts = mask3.sum(dim=2)                 # (B,S)
    is_real = tok_counts > 0
    nll_sum = per_tok.sum(dim=2)                  # (B,S)
    nll_avg = nll_sum / (tok_counts + eps)

    return nll_avg, is_real

In [ ]:
def sample_length_matched_indices(lengths, valid_mask, tol=0.10):
    """
    lengths: (B,) token counts
    valid_mask: (B,) bool (must be True for selectable negatives)
    Returns neg_idx: (B,) where neg_idx[i] != i and valid_mask[neg_idx[i]] = True
    """
    device = lengths.device
    B = lengths.size(0)
    neg_idx = torch.empty(B, dtype=torch.long, device=device)

    all_idx = torch.arange(B, device=device)

    for i in range(B):
        if not valid_mask[i]:
            # doesn't matter; will be ignored by is_real anyway
            # pick something valid to avoid crash
            candidates = all_idx[valid_mask]
            neg_idx[i] = candidates[0] if len(candidates) else i
            continue

        li = lengths[i].float()
        # candidates: valid, not self, length within tol
        candidates = all_idx[
            valid_mask &
            (all_idx != i) &
            (torch.abs(lengths.float() - li) <= tol * torch.clamp(li, min=1.0))
        ]

        if len(candidates) == 0:
            # fallback: any other valid
            candidates = all_idx[valid_mask & (all_idx != i)]

        neg_idx[i] = candidates[torch.randint(0, len(candidates), (1,), device=device)] if len(candidates) else i

    return neg_idx

In [ ]:
def test_latent_discrimination_hardneg(encoder, decoder, dataloader, device,
                                       num_negatives=5, tol=0.10, use_mu=True):
    encoder.eval()
    decoder.eval()

    total = 0
    correct = 0

    with torch.no_grad():
        for batch in dataloader:
            enc_input_ids = batch["enc_input_ids"].to(device)
            enc_word_mask = batch["enc_word_mask"].to(device)
            dec_input_ids = batch["dec_input_ids"].to(device)
            dec_word_mask = batch["dec_word_mask"].to(device)

            B, S, W = dec_input_ids.shape

            mu_t, sigma2_t, mu_i, sigma2_i = encoder(dec_input_ids, dec_word_mask)
            z_t = mu_t if use_mu else reparameterize(mu_t, sigma2_t)
            z_i = mu_i if use_mu else reparameterize(mu_i, sigma2_i)

            pos_logits, _, _ = decoder(dec_input_ids, dec_word_mask, z_t, z_i)
            pos_nll, is_real = compute_sentence_nll_avg(pos_logits, dec_input_ids, dec_word_mask)

            lengths = dec_word_mask.sum(dim=2)  # (B,S)

            for _ in range(num_negatives):
                neg_ids = dec_input_ids.clone()
                neg_mask = dec_word_mask.clone()

                # For each sentence slot s, length-match within that slot
                for s in range(S):
                    lens_s = lengths[:, s]
                    valid_s = lens_s > 0
                    idx_s = sample_length_matched_indices(lens_s, valid_s, tol=tol)
                    neg_ids[:, s, :] = dec_input_ids[idx_s, s, :]
                    neg_mask[:, s, :] = dec_word_mask[idx_s, s, :]

                neg_logits, _, _ = decoder(neg_ids, neg_mask, z_t, z_i)
                neg_nll, _ = compute_sentence_nll_avg(neg_logits, neg_ids, neg_mask)

                wins = (pos_nll < neg_nll) & is_real
                correct += wins.sum().item()
                total += is_real.sum().item()

    acc = correct / max(total, 1)
    print(f"HardNeg Discrimination Acc: {acc:.2%} (comparisons: {total})")
    return acc

In [ ]:
test_latent_discrimination_hardneg(inference_net, generative_net, train_loader, device,
                                       num_negatives=5, tol=0.10, use_mu=False)

In [ ]:
# Discriminate between two sentences of the same abstract/answer
# -------------------------
# Helpers
# -------------------------

def _sentence_prefix_key(ids_1d, mask_1d, n_prefix=3, bos_id=50256, skip_bos=True):
    """
    ids_1d, mask_1d: (W,)
    Returns a hashable key = first n_prefix real tokens (optionally skipping BOS if it's first real token).
    """
    real = ids_1d[mask_1d.bool()]
    if real.numel() == 0:
        return ("<EMPTY>",)

    if skip_bos and int(real[0].item()) == bos_id:
        real = real[1:]

    if real.numel() == 0:
        return ("<EMPTY_AFTER_BOS>",)

    real = real[:n_prefix]
    return tuple(real.detach().cpu().tolist())


def _collect_valid_sentence_pairs(dec_word_mask):
    """
    dec_word_mask: (B,S,W)
    Returns list of (b,s) pairs where sentence length > 0.
    """
    lens = dec_word_mask.sum(dim=2)  # (B,S)
    bs = torch.nonzero(lens > 0, as_tuple=False)  # (M,2)
    return [(int(b.item()), int(s.item())) for b, s in bs]


def _answer_has_any_token(dec_word_mask):
    # (B,) bool
    return (dec_word_mask.sum(dim=(1, 2)) > 0)


def _sentence_lengths(dec_word_mask):
    # (B,S) long
    return dec_word_mask.sum(dim=2).long()


def _margin_stats(margins: list):
    if len(margins) == 0:
        return {"count": 0}
    x = torch.tensor(margins, dtype=torch.float32)
    return {
        "count": int(x.numel()),
        "mean": float(x.mean().item()),
        "median": float(x.median().item()),
        "p25": float(x.kthvalue(max(1, int(0.25 * x.numel()))).values.item()),
        "p75": float(x.kthvalue(max(1, int(0.75 * x.numel()))).values.item()),
        "win_rate": float((x > 0).float().mean().item()),
    }


# -------------------------
# TEST 1: Intra-answer hard negatives
# -------------------------

@torch.no_grad()
def test_sentence_plan_discrimination_intra_answer(
    encoder, decoder, dataloader, device,
    num_negatives=5,
    use_mu=True,
    length_match=True,
    tol=0.10,
    bos_id=50256,
    skip_bos=True,
    max_batches=None,
):
    """
    For each (b,s) real sentence:
      pos = sentence tokens at slot s
      neg = a DIFFERENT sentence slot s2 != s within SAME answer b (hard negative)
    Compare avg NLL/token at slot s under (z_t[b], z_i[b,s]) with pos vs neg.

    This tests: does local plan z_i[b,s] identify its own sentence vs other sentences in same answer?
    """
    encoder.eval(); decoder.eval()
    total = 0
    correct = 0
    margins = []

    for bi, batch in enumerate(dataloader):
        if max_batches is not None and bi >= max_batches:
            break

        dec_input_ids = batch["dec_input_ids"].to(device)   # (B,S,W)
        dec_word_mask = batch["dec_word_mask"].to(device)   # (B,S,W)
        B, S, W = dec_input_ids.shape

        # Encode once on the ORIGINAL answer
        mu_t, sigma2_t, mu_i, sigma2_i = encoder(dec_input_ids, dec_word_mask)
        z_t = mu_t if use_mu else reparameterize(mu_t, sigma2_t)
        z_i = mu_i if use_mu else reparameterize(mu_i, sigma2_i)



        # Positive scores
        pos_logits, _, _ = decoder(dec_input_ids, dec_word_mask, z_t, z_i)
        pos_nll_sent, is_real = compute_sentence_nll_avg(pos_logits, dec_input_ids, dec_word_mask)  # (B,S),(B,S)

        lens = _sentence_lengths(dec_word_mask)  # (B,S)

        # Precompute valid sentence slots per answer b
        valid_slots = []
        for b in range(B):
            slots = torch.nonzero(lens[b] > 0, as_tuple=False).squeeze(-1).tolist()
            valid_slots.append(slots)

        for _ in range(num_negatives):
            neg_ids = dec_input_ids.clone()
            neg_msk = dec_word_mask.clone()

            # choose s2 within same b for each s
            for b in range(B):
                slots = valid_slots[b]
                if len(slots) < 2:
                    continue  # can't form intra-answer negative

                for s in slots:
                    # candidate negatives: other slots
                    cand = [s2 for s2 in slots if s2 != s]

                    # optional length-match within tol
                    if length_match:
                        ls = float(lens[b, s].item())
                        cand2 = [s2 for s2 in cand if abs(float(lens[b, s2].item()) - ls) <= tol * max(ls, 1.0)]
                        cand = cand2 if len(cand2) > 0 else cand

                    s2 = cand[random.randrange(len(cand))]

                    # replace sentence slot s with sentence s2 tokens/mask (same answer)
                    neg_ids[b, s, :] = dec_input_ids[b, s2, :]
                    neg_msk[b, s, :] = dec_word_mask[b, s2, :]

            # Score negatives under SAME latents (z_t,z_i from original)
            neg_logits, _, _ = decoder(neg_ids, neg_msk, z_t, z_i)
            neg_nll_sent, _ = compute_sentence_nll_avg(neg_logits, neg_ids, neg_msk)  # (B,S)

            # Only count slots that were real in the positive
            wins = (pos_nll_sent < neg_nll_sent) & is_real
            correct += wins.sum().item()
            total += is_real.sum().item()

            delta = (neg_nll_sent - pos_nll_sent)[is_real]
            margins.extend(delta.detach().cpu().tolist())

    acc = correct / max(total, 1)
    stats = _margin_stats(margins)
    print(f"[Intra-answer sentence neg] Acc: {acc:.2%}  comps={total}  margin={stats}")
    return acc, stats




In [ ]:
#sample k sentences and compare with the one with lowest nll
@torch.no_grad()
def _build_prefix_embeds_from_latents(decoder, z_t, z_i_samples):
    """
    Reproduces your decoder's plan generation + prefix_generator part.
    Returns:
      prefix_embeds_flat: (B*S, prefix_len, gpt_hidden)
    """
    B, S, _ = z_i_samples.shape

    z_t_expanded = z_t.unsqueeze(1).expand(B, S, z_t.size(-1))
    gru_input = torch.cat([z_i_samples, z_t_expanded], dim=-1)  # (B,S, local+global)

    h0 = decoder.gru_initial_projection(z_t).unsqueeze(0)       # (1,B,H)
    h0 = h0.repeat(decoder.gru.num_layers, 1, 1)                # (L,B,H)

    plan_vectors, _ = decoder.gru(gru_input, h0)                # (B,S,d_model)
    flat_plan_vectors = plan_vectors.reshape(-1, decoder.d_model)  # (B*S, d_model)


    return flat_plan_vectors


@torch.no_grad()
def _nll_from_prefix_pairs(
    decoder,
    compute_sentence_nll_avg,
    input_ids_flat,      # (N, W)
    word_mask_flat,      # (N, W)
    plan_flat,           # (N, d_model) - Assumed to be the latent vectors
    pair_chunk=256,
):
    """
    Runs the GPT2 forward with Deep Latent Injection (KV Prefix + Embedding Bias).
    Returns:
      nll: (N,)
      is_real: (N,)
    """
    device = input_ids_flat.device
    N, W = input_ids_flat.shape

    nll_out = torch.empty((N,), device=device, dtype=torch.float32)
    real_out = torch.empty((N,), device=device, dtype=torch.bool)

    for start in range(0, N, pair_chunk):
        end = min(start + pair_chunk, N)
        m = end - start

        ids = input_ids_flat[start:end]     # (m, W)
        msk = word_mask_flat[start:end]     # (m, W)
        curr_plan = plan_flat[start:end]    # (m, d_model)

        # 1. Prepare Plan (Norm & Projections)
        norm_plan = decoder.plan_ln(curr_plan)

        # A. Calculate Embedding Bias (Gate * Plan)
        # Shape: (m, 1, d_model) -> Broadcasts to all W tokens
        plan_emb = decoder.plan_to_emb(norm_plan)
        gate = decoder.plan_gate(norm_plan)
        emb_bias = (gate * plan_emb).unsqueeze(1)

        # B. Calculate KV Prefix (Past)
        kv = decoder.plan_to_kv(norm_plan)
        kv = kv.view(
            m,
            decoder.gpt2_n_layer,
            2,
            decoder.gpt2_n_head,
            decoder.prefix_len,
            decoder.gpt2_head_dim
        )
        kv = kv.permute(1, 2, 0, 3, 4, 5).contiguous()

        # Initialize Cache with Plan
        past = DynamicCache()
        for l in range(decoder.gpt2_n_layer):
            past.update(kv[l, 0], kv[l, 1], l)

        # 2. Prepare Inputs (Manual Embedding Injection)
        # Get raw token embeddings
        tok_emb = decoder.gpt2_model.wte(ids) # (m, W, d_model)
        # Add bias to every token in the sequence
        tok_emb = tok_emb + emb_bias

        # 3. Prepare Attention Mask
        # We must mask both the Prefix (already in 'past') and the Inputs
        prefix_mask = torch.ones(
            (m, decoder.prefix_len),
            device=device,
            dtype=msk.dtype
        )
        attn_mask = torch.cat([prefix_mask, msk], dim=1) # (m, P + W)

        # 4. Forward Pass
        # Returns hidden states for the input tokens only (W), attending to 'past'
        gpt2_output = decoder.gpt2_model.transformer(
            inputs_embeds=tok_emb,
            attention_mask=attn_mask,
            past_key_values=past,
            use_cache=True
        )

        text_hidden = gpt2_output.last_hidden_state # (m, W, H)
        logits = decoder.final_linear(text_hidden) # (m, W, V)

        # 5. Compute NLL
        # compute_sentence_nll_avg expects (B, S, W, V) -> use S=1
        logits_4d = logits.unsqueeze(1)     # (m, 1, W, V)
        ids_3d   = ids.unsqueeze(1)         # (m, 1, W)
        msk_3d   = msk.unsqueeze(1)         # (m, 1, W)

        nll, is_real = compute_sentence_nll_avg(logits_4d, ids_3d, msk_3d)
        nll = nll.squeeze(1)                # (m,)
        is_real = is_real.squeeze(1)        # (m,)

        nll_out[start:end] = nll
        real_out[start:end] = is_real

    return nll_out, real_out

@torch.no_grad()
def latent_discrimination_hardest_of_k(
    encoder,
    decoder,
    dataloader,
    device,
    compute_sentence_nll_avg,
    reparameterize_fn=None,   # pass your reparameterize if use_mu=False
    use_mu=True,
    K=50,
    max_comparisons=10_000,
    anchor_chunk=16,          # anchors processed together
    pair_chunk=256,           # pairs per GPT forward (memory knob)
    seed=0,
):
    """
    For each anchor i in a batch:
      pos = NLL(sentence_i | plan_i)
      negs: sample K other sentences j, compute NLL(sentence_j | plan_i)
      hardest_neg = min_j neg
      win if pos < hardest_neg
      margin = hardest_neg - pos

    Stops after max_comparisons anchors across batches.
    """
    encoder.eval()
    decoder.eval()

    gen = torch.Generator(device="cpu")
    gen.manual_seed(seed)

    total = 0
    wins = 0
    ties = 0

    margins = []
    pos_nlls = []
    hard_neg_nlls = []

    for batch in dataloader:
        if total >= max_comparisons:
            break

        # --- fetch inputs (adjust keys if your batch dict differs) ---
        dec_input_ids = batch["dec_input_ids"].to(device)     # (B,S,W)
        dec_word_mask = batch["dec_word_mask"].to(device)     # (B,S,W)

        B, S, W = dec_input_ids.shape
        N = B * S

        # --- encode latents ---
        mu_t, sigma2_t, mu_i, sigma2_i = encoder(dec_input_ids, dec_word_mask)
        if use_mu:
            z_t = mu_t
            z_i = mu_i
        else:
            assert reparameterize_fn is not None, "Pass reparameterize_fn if use_mu=False"
            z_t = reparameterize_fn(mu_t, sigma2_t)
            z_i = reparameterize_fn(mu_i, sigma2_i)

        # --- POS NLL: run your decoder normally (correct sentence + correct plan) ---
        pos_logits, _, _ = decoder(dec_input_ids, dec_word_mask, z_t, z_i)
        pos_nll, is_real = compute_sentence_nll_avg(pos_logits, dec_input_ids, dec_word_mask)  # (B,S), (B,S)

        # flatten
        ids_flat = dec_input_ids.view(N, W)
        msk_flat = dec_word_mask.view(N, W)
        pos_nll_flat = pos_nll.view(N)
        is_real_flat = is_real.view(N)

        real_idx = torch.nonzero(is_real_flat, as_tuple=False).squeeze(1).cpu()
        P = int(real_idx.numel())
        if P < 2:
            continue

        # --- build all prefixes for this batch plans (plan_i -> prefix_i) ---
        plans_flat = _build_prefix_embeds_from_latents(decoder, z_t, z_i)  # (N, prefix_len, H)

        # randomize anchor order
        perm = torch.randperm(P, generator=gen)
        anchors_all = real_idx[perm]  # CPU indices into [0..N)

        # process anchors in chunks
        ptr = 0
        while ptr < P and total < max_comparisons:
            A = min(anchor_chunk, P - ptr, max_comparisons - total)
            anchors = anchors_all[ptr:ptr + A]  # (A,) on CPU
            ptr += A

            anchors_dev = anchors.to(device)

            # sample K negatives (with replacement), excluding anchor itself
            pool = real_idx  # CPU
            pool_dev = pool.to(device)
            Ppool = pool.numel()

            # start with random picks in pool
            picks = torch.randint(0, Ppool, (A, K), generator=gen)  # CPU
            neg_idx = pool[picks]                                    # CPU indices into [0..N)
            # enforce neg != anchor (resample offending positions)
            neq = (neg_idx != anchors.view(-1, 1))
            while not bool(neq.all()):
                bad = torch.nonzero(~neq, as_tuple=False)
                newp = torch.randint(0, Ppool, (bad.size(0),), generator=gen)
                neg_idx[bad[:, 0], bad[:, 1]] = pool[newp]
                neq = (neg_idx != anchors.view(-1, 1))

            neg_idx_dev = neg_idx.to(device)  # (A,K)

            # build (A*K) PAIRS: (sentence_neg, prefix_anchor)
            ids_pairs = ids_flat[neg_idx_dev.reshape(-1)]  # (A*K, W)
            msk_pairs = msk_flat[neg_idx_dev.reshape(-1)]  # (A*K, W)

            plans_anchor = plans_flat[anchors_dev]                     # (A, P, H)
            plan_pairs = plans_anchor.unsqueeze(1).expand(A, K, -1, -1) # (A,K,P,H)
            plan_pairs = plan_pairs.reshape(A * K, plans_anchor.size(1), plans_anchor.size(2))

            neg_nll_pairs, _ = _nll_from_prefix_pairs(
                decoder,
                compute_sentence_nll_avg,
                ids_pairs,
                msk_pairs,
                plan_pairs,
                pair_chunk=pair_chunk,
            )
            neg_nll = neg_nll_pairs.view(A, K)               # (A,K)
            hard_neg = neg_nll.min(dim=1).values             # (A,)

            pos = pos_nll_flat[anchors_dev]                  # (A,)
            margin = (hard_neg - pos)                        # (A,)

            wins += int((margin > 0).sum().item())
            ties += int((margin == 0).sum().item())
            total += A

            margins.append(margin.detach().cpu())
            pos_nlls.append(pos.detach().cpu())
            hard_neg_nlls.append(hard_neg.detach().cpu())

    if total == 0:
        return {
            "total": 0,
            "accuracy": float("nan"),
            "wins": 0,
            "ties": 0,
            "margin_p25": float("nan"),
            "margin_median": float("nan"),
            "margin_p75": float("nan"),
        }

    margins = torch.cat(margins)
    pos_nlls = torch.cat(pos_nlls)
    hard_neg_nlls = torch.cat(hard_neg_nlls)

    accuracy = wins / total
    q25, q50, q75 = torch.quantile(margins, torch.tensor([0.25, 0.50, 0.75])).tolist()

    return {
        "total": total,
        "wins": wins,
        "ties": ties,
        "accuracy": accuracy,
        "mean_pos_nll": float(pos_nlls.mean().item()),
        "mean_hard_neg_nll": float(hard_neg_nlls.mean().item()),
        "mean_margin": float(margins.mean().item()),
        "margin_p25": float(q25),
        "margin_median": float(q50),
        "margin_p75": float(q75),
    }

In [ ]:
# get sbert embedding of the sentences
def sbert_embedder_wrapper(sbert_model, dec_tokenizer):
    """
    Creates an embed_fn that:
      1. Decodes the input IDs back to text strings (using dec_tokenizer).
      2. Re-encodes them using the SBERT model to get high-quality embeddings.
    """

    def embed_fn(ids_2d: torch.Tensor, mask_2d: torch.Tensor) -> torch.Tensor:
        # ids_2d: (N, W)

        # 1. Convert IDs -> Text (The Bridge)
        # We perform this on CPU because decode works on lists/strings
        ids_list = ids_2d.detach().cpu().tolist()

        # skip_special_tokens=True removes the padding and start/end tokens automatically
        sentences = dec_tokenizer.batch_decode(ids_list, skip_special_tokens=True)

        # 2. Encode with SBERT
        # convert_to_tensor=True gives us a torch tensor on the correct device
        embeddings = sbert_model.encode(sentences, convert_to_tensor=True, show_progress_bar=False)

        return embeddings # Shape: (N, SBERT_Hidden_Dim)

    return embed_fn

In [ ]:
# sbert hard neg discrimination
@torch.no_grad()
def _embed_in_chunks(embed_fn, ids_2d, mask_2d, chunk=256):
    """
    embed_fn: (N,W), (N,W) -> (N,D)
    Returns embeddings on the same device as ids_2d.
    """
    device = ids_2d.device
    outs = []
    for start in range(0, ids_2d.size(0), chunk):
        end = min(start + chunk, ids_2d.size(0))
        emb = embed_fn(ids_2d[start:end], mask_2d[start:end])
        if not isinstance(emb, torch.Tensor):
            emb = torch.tensor(emb)
        emb = emb.to(device)
        outs.append(emb)
    return torch.cat(outs, dim=0)


@torch.no_grad()
def latent_discrimination_sbert_closest_of_k(
    encoder,
    decoder,
    dataloader,
    device,
    compute_sentence_nll_avg,
    embed_fn,                  # from sbert_embedder_wrapper(...)
    reparameterize_fn=None,     # your reparameterize if use_mu=False
    use_mu=True,
    K=50,
    max_comparisons=10_000,
    anchor_chunk=32,
    pair_chunk=256,            # passed to _nll_from_prefix_pairs
    embed_chunk=256,           # SBERT batching
    seed=0,
):
    """
    For each anchor i (real sentence) in a batch:
      pos = NLL(sentence_i | plan_i)
      sample K negatives j from (B*S real sentences)
      pick closest negative by SBERT cosine similarity: argmax cos(sbert(i), sbert(j))
      neg = NLL(sentence_j | plan_i)
      win if pos < neg
      margin = neg - pos

    Stops after max_comparisons anchors across dataloader.
    """

    encoder.eval()
    decoder.eval()

    gen = torch.Generator(device="cpu")
    gen.manual_seed(seed)

    total = 0
    wins = 0
    ties = 0

    margins = []
    pos_nlls = []
    chosen_neg_nlls = []
    chosen_cos_sims = []

    for batch in dataloader:
        if total >= max_comparisons:
            break

        dec_input_ids = batch["dec_input_ids"].to(device)   # (B,S,W)
        dec_word_mask = batch["dec_word_mask"].to(device)   # (B,S,W)

        B, S, W = dec_input_ids.shape
        N = B * S

        # --- encode latents ---
        mu_t, sigma2_t, mu_i, sigma2_i = encoder(dec_input_ids, dec_word_mask)
        if use_mu:
            z_t = mu_t
            z_i = mu_i
        else:
            assert reparameterize_fn is not None, "Pass reparameterize_fn if use_mu=False"
            z_t = reparameterize_fn(mu_t, sigma2_t)
            z_i = reparameterize_fn(mu_i, sigma2_i)

        # --- POS NLL (correct sentence + correct plan) ---
        pos_logits, _, _ = decoder(dec_input_ids, dec_word_mask, z_t, z_i)
        pos_nll, is_real = compute_sentence_nll_avg(pos_logits, dec_input_ids, dec_word_mask)  # (B,S)

        ids_flat = dec_input_ids.view(N, W)
        msk_flat = dec_word_mask.view(N, W)
        pos_nll_flat = pos_nll.view(N)
        is_real_flat = is_real.view(N)

        real_idx = torch.nonzero(is_real_flat, as_tuple=False).squeeze(1).cpu()  # global indices in [0..N)
        P = int(real_idx.numel())
        if P < 2:
            continue

        # --- prefixes for plans (plan_i -> prefix_i), for all sentences ---
        plans_flat = _build_prefix_embeds_from_latents(decoder, z_t, z_i)  # (N, prefix_len, H)

        # --- SBERT embeddings for ALL real sentences (once per batch) ---
        real_idx_dev = real_idx.to(device)
        real_ids = ids_flat[real_idx_dev]     # (P,W)
        real_msk = msk_flat[real_idx_dev]     # (P,W)
        real_emb = _embed_in_chunks(embed_fn, real_ids, real_msk, chunk=embed_chunk)  # (P,D)

        # normalize once for cosine similarity
        real_emb = F.normalize(real_emb, dim=-1)

        # mapping: global sentence index -> row in real_emb
        index_map = torch.full((N,), -1, dtype=torch.long, device="cpu")
        index_map[real_idx] = torch.arange(P, dtype=torch.long)

        # random anchor order
        perm = torch.randperm(P, generator=gen)
        anchors_all = real_idx[perm]  # CPU global indices

        ptr = 0
        while ptr < P and total < max_comparisons:
            A = min(anchor_chunk, P - ptr, max_comparisons - total)
            anchors = anchors_all[ptr:ptr + A]  # CPU global indices
            ptr += A

            # sample K negatives from real pool, excluding anchor itself
            picks = torch.randint(0, P, (A, K), generator=gen)   # CPU indices into [0..P)
            neg_rows = picks.clone()                              # rows in real_emb
            anchor_rows = index_map[anchors]                      # (A,) rows in real_emb

            # enforce row != anchor_row
            neq = (neg_rows != anchor_rows.view(-1, 1))
            while not bool(neq.all()):
                bad = torch.nonzero(~neq, as_tuple=False)
                neg_rows[bad[:, 0], bad[:, 1]] = torch.randint(0, P, (bad.size(0),), generator=gen)
                neq = (neg_rows != anchor_rows.view(-1, 1))

            # cosine similarity: choose closest (max cos)
            a_emb = real_emb[anchor_rows.to(device)]              # (A,D)
            n_emb = real_emb[neg_rows.to(device)]                 # (A,K,D)
            cos = (n_emb * a_emb.unsqueeze(1)).sum(dim=-1)        # (A,K)
            best_k = cos.argmax(dim=1)                            # (A,)

            # chosen negative global indices:
            # we sampled negatives by rows-in-real_emb; convert row->global via real_idx[row]
            chosen_neg_rows = neg_rows[torch.arange(A), best_k.cpu()]  # CPU rows
            chosen_neg_global = real_idx[chosen_neg_rows]              # CPU global idx in [0..N)

            # compute NLL(sentence_neg | plan_anchor)
            anchors_dev = anchors.to(device)
            chosen_neg_dev = chosen_neg_global.to(device)

            ids_pairs = ids_flat[chosen_neg_dev]     # (A,W)
            msk_pairs = msk_flat[chosen_neg_dev]     # (A,W)
            plan_pairs = plans_flat[anchors_dev]    # (A,prefix_len,H)

            neg_nll, _ = _nll_from_prefix_pairs(
                decoder,
                compute_sentence_nll_avg,
                ids_pairs,
                msk_pairs,
                plan_pairs,
                pair_chunk=pair_chunk,
            )  # (A,)

            pos = pos_nll_flat[anchors_dev]          # (A,)
            margin = (neg_nll - pos)                 # (A,)

            wins += int((margin > 0).sum().item())
            ties += int((margin == 0).sum().item())
            total += A

            margins.append(margin.detach().cpu())
            pos_nlls.append(pos.detach().cpu())
            chosen_neg_nlls.append(neg_nll.detach().cpu())
            chosen_cos_sims.append(cos.max(dim=1).values.detach().cpu())

    if total == 0:
        return {
            "total": 0,
            "accuracy": float("nan"),
            "wins": 0,
            "ties": 0,
            "margin_p25": float("nan"),
            "margin_median": float("nan"),
            "margin_p75": float("nan"),
        }

    margins = torch.cat(margins)
    pos_nlls = torch.cat(pos_nlls)
    chosen_neg_nlls = torch.cat(chosen_neg_nlls)
    chosen_cos_sims = torch.cat(chosen_cos_sims)

    accuracy = wins / total
    q25, q50, q75 = torch.quantile(margins, torch.tensor([0.25, 0.50, 0.75])).tolist()

    return {
        "total": total,
        "wins": wins,
        "ties": ties,
        "accuracy": accuracy,
        "mean_pos_nll": float(pos_nlls.mean().item()),
        "mean_chosen_neg_nll": float(chosen_neg_nlls.mean().item()),
        "mean_margin": float(margins.mean().item()),
        "margin_p25": float(q25),
        "margin_median": float(q50),
        "margin_p75": float(q75),
        "mean_chosen_cos_sim": float(chosen_cos_sims.mean().item()),
    }

In [ ]:
# model hardk disc ran here
klist = [5,10,20,50,80]
for k in klist:
  results = latent_discrimination_hardest_of_k(
    inference_net,
    generative_net,
    train_loader,
    device,
    compute_sentence_nll_avg,
    reparameterize_fn=None,   # pass your reparameterize if use_mu=False
    use_mu=True,
    K=k,
    max_comparisons=10_000,
    anchor_chunk=16,          # anchors processed together
    pair_chunk=256,           # pairs per GPT forward (memory knob)
    seed=0,
)
  print(f"K: {k} ",results)

In [ ]:
# sbert hardk disc ran here
sbert = SentenceTransformer('all-MiniLM-L6-v2')
embed_fn = sbert_embedder_wrapper(sbert, dec_tokenizer)

for k in klist:

  stats = latent_discrimination_sbert_closest_of_k(
      encoder=inference_net,
      decoder=generative_net,
      dataloader=train_loader,
      device=device,
      compute_sentence_nll_avg=compute_sentence_nll_avg,
      embed_fn=embed_fn,
      reparameterize_fn=reparameterize,  # only used if use_mu=False
      use_mu=True,
      K=k,
      max_comparisons=10_000,
  )
  print(f"K : {k},  {stats}")

In [ ]:
# intra answer disc ran here
test_sentence_plan_discrimination_intra_answer(
    inference_net, generative_net, train_loader, device,
    num_negatives=5,
    use_mu=True,
    length_match=True,
    tol=0.10,
    bos_id=50256,
    skip_bos=True,
    max_batches=None,
)

In [ ]:
def greedy_decode_with_oracle_prefix(decoder, curr_plans, gold_ids, gold_mask,
                                    K, max_new_tokens=200,
                                    bos_id=50256, eos_id=50256):
    """
    Returns token-accuracy over the remaining gold tokens after K.
    """
    device = gold_ids.device

    # --- FIX 1: Define Batch Size 'm' ---
    m = curr_plans.size(0)

    L = int(gold_mask.sum().item())
    if L <= 1:
        return None  # too short

    gold = gold_ids[:L]  # (L,)
    K = int(min(K, L-1))  # ensure at least 1 target token remains

    # Target to evaluate against
    target = gold[K:]
    if target.numel() == 0:
        return 0.0 # No tokens to predict

    # Context: [BOS] + gold[:K]
    ctx_ids = torch.cat([torch.tensor([bos_id], device=device), gold[:K]], dim=0).unsqueeze(0)  # (1, 1+K)

    # 1. Prepare Plan & KV (Deep Injection)
    norm_plan = decoder.plan_ln(curr_plans)

    # A. Embedding Bias
    plan_emb = decoder.plan_to_emb(norm_plan)
    gate = decoder.plan_gate(norm_plan)
    emb_bias = (gate * plan_emb).unsqueeze(1) # (m, 1, H)

    # B. KV Prefix
    kv = decoder.plan_to_kv(norm_plan)
    kv = kv.view(
        m,
        decoder.gpt2_n_layer,
        2,
        decoder.gpt2_n_head,
        decoder.prefix_len,
        decoder.gpt2_head_dim
    )
    kv = kv.permute(1, 2, 0, 3, 4, 5).contiguous()

    past = DynamicCache()
    for l in range(decoder.gpt2_n_layer):
        past.update(kv[l, 0], kv[l, 1], l)

    # 2. Initial Forward Pass (Process Oracle Context)
    tok_emb = decoder.gpt2_model.wte(ctx_ids) # (m, 1+K, H)
    tok_emb = tok_emb + emb_bias

    gpt2_output = decoder.gpt2_model.transformer(
        inputs_embeds=tok_emb,
        past_key_values=past,
        use_cache=True,
        return_dict=True
    )

    # --- FIX 2: Capture initial 'past' ---
    past = gpt2_output.past_key_values

    last_h = gpt2_output.last_hidden_state[:,-1,:]

    preds = []
    # Predict first token
    logits = decoder.final_linear(last_h)
    next_tok = torch.argmax(logits, dim=-1)
    preds.append(next_tok.item())

    # 3. Generation Loop
    T = target.size(0)

    # We loop until we match target length or max tokens
    for _ in range(min(max_new_tokens, T-1)):
        if preds[-1] == eos_id:
            break

        # --- FIX 3: Correct Input Shape (1,1) ---
        ctx_ids = torch.tensor([[preds[-1]]], device=device, dtype=torch.long) # Shape (1, 1)

        tok_emb = decoder.gpt2_model.wte(ctx_ids)
        tok_emb = tok_emb + emb_bias

        gpt2_output = decoder.gpt2_model.transformer(
            inputs_embeds=tok_emb,
            past_key_values=past, # Pass history
            use_cache=True,
            return_dict=True
        )

        # --- FIX 4: Update 'past' ---
        past = gpt2_output.past_key_values

        last_h = gpt2_output.last_hidden_state[:, -1, :]
        logits = decoder.final_linear(last_h)
        next_tok = torch.argmax(logits, dim=-1)
        preds.append(next_tok.item())

    # Score
    preds_t = torch.tensor(preds, device=device, dtype=torch.long)
    m_len = min(preds_t.numel(), target.numel())
    correct = (preds_t[:m_len] == target[:m_len]).sum().item()
    total = target.numel()

    return correct / total

In [ ]:
def oracle_prefix_sweep(encoder, decoder, dataloader, device,
                        K_list=(0,5,10,20,40,80),
                        max_new_tokens=200, num_batches=20,
                        use_mu=True, bos_id=50256, eos_id=50256):
    encoder.eval()
    decoder.eval()

    K_list = list(K_list)
    acc_sum = {K: 0.0 for K in K_list}
    acc_cnt = {K: 0   for K in K_list}

    with torch.no_grad():
        for b_idx, batch in enumerate(dataloader):
            if b_idx >= num_batches:
                break

            enc_input_ids = batch["enc_input_ids"].to(device)
            enc_word_mask = batch["enc_word_mask"].to(device)
            dec_input_ids = batch["dec_input_ids"].to(device)
            dec_word_mask = batch["dec_word_mask"].to(device)

            B, S, W = dec_input_ids.shape

            mu_t, sigma2_t, mu_i, sigma2_i = encoder(dec_input_ids, dec_word_mask)
            z_t = mu_t if use_mu else reparameterize(mu_t, sigma2_t)
            z_i = mu_i if use_mu else reparameterize(mu_i, sigma2_i)

            # Compute plan vectors like your inference
            z_t_exp = z_t.unsqueeze(1).repeat(1, S, 1)
            gru_in = torch.cat([z_i, z_t_exp], dim=-1)
            h0 = decoder.gru_initial_projection(z_t).unsqueeze(0).repeat(decoder.gru.num_layers, 1, 1)
            plan_vecs, _ = decoder.gru(gru_in, h0)  # (B,S,D_model)

            for i in range(B):
                for s in range(S):
                    # skip padding sentences
                    if dec_word_mask[i, s].sum().item() == 0:
                        continue

                    plan = plan_vecs[i, s].unsqueeze(0)  # (1,D)


                    gold_ids = dec_input_ids[i, s]   # (W,)
                    gold_mask = dec_word_mask[i, s]  # (W,)

                    for K in K_list:
                        a = greedy_decode_with_oracle_prefix(
                            decoder, plan, gold_ids, gold_mask,
                            K=K, max_new_tokens=max_new_tokens,
                            bos_id=bos_id, eos_id=eos_id
                        )
                        if a is None:
                            continue
                        acc_sum[K] += a
                        acc_cnt[K] += 1

    results = {K: (acc_sum[K] / max(acc_cnt[K], 1)) for K in K_list}
    print("Oracle prefix-length curve (avg token-acc on remaining tokens):")
    for K in K_list:
        print(f"  K={K:>3}: acc={results[K]:.4f}  (n={acc_cnt[K]})")
    return results

In [ ]:
doc_eos = dec_tokenizer.eos_token_id                 # DOCUMENT end
bos_id = dec_tokenizer.bos_token_id
pad_token_id = dec_tokenizer.pad_token_id
sent_eos = dec_tokenizer.convert_tokens_to_ids("<EOS>")  # SENTENCE end
oracle_prefix_sweep(inference_net, generative_net, train_loader, device,
                        K_list=(0,5,10,20,25,30,40,45),
                        max_new_tokens=50, num_batches=20,
                        use_mu=True, bos_id=bos_id, eos_id=sent_eos)